# Validate Response and Catalog Models

Tests the adaptive conclusions under response perturbations and audits catalog-passage recovery using frozen analysis settings.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import gzip
import hashlib
import json
import math
import os
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.optimize import minimize
from scipy.special import gammaln
from scipy.stats import chisquare, nbinom, norm

try:
    from sgp4.api import Satrec, SatrecArray
except ImportError:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', 'sgp4>=2.25,<3'],
        check=True,
    )
    from sgp4.api import Satrec, SatrecArray

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


FINE_GRID_DIR = PROJECT_ROOT / 'phase1_outputs' / 'ordem_action_library' / 'b982_fine_elevation_49'
RESPONSE_PATH = FINE_GRID_DIR / 'ordem_fine_elevation_response_matrix_49.csv'
RESPONSE_UNCERTAINTY_PATH = FINE_GRID_DIR / 'ordem_refinement_action_uncertainty.csv'
ASSET_PATH = PROJECT_ROOT / 'phase2_outputs' / 'dynamic_asset_exposure' / 'active_omm_action_audit_v1' / 'active_satellite_shell_exposure.csv'
CATALOG_DIR = PROJECT_ROOT / 'phase0_outputs' / 'catalog_matching' / 'ca_cfar_1p75_24h_final_audited'
EVENT_PATH = CATALOG_DIR / 'final_consolidated_catalog_association.csv'
HISTORY_PATH = PROJECT_ROOT / 'phase0_outputs' / 'catalog_matching' / 'historical_catalog' / 'gp_history_20180103_20180106.json.gz'
PREVIOUS_DIR = PROJECT_ROOT / 'phase3_outputs' / 'final_bayesian_policy' / 'fine_grid_4fa819596ffc'
PREVIOUS_CONFIG_PATH = PREVIOUS_DIR / 'final_experiment_config.json'
REAL_COUNT_CV_PATH = PREVIOUS_DIR / 'real_hourly_count_blocked_cv.csv'

EXPECTED_SHA256 = {
    'response': '1d06d7faaf496a037c15a2bbe5ba5b72f5bb6adbdc6be061202d5cafd6ab0566',
    'response_uncertainty': 'dd3ead00190e0034d61308cd0d65e64d50bf9d1bf8059834908ec023f4cae08c',
    'assets': '49e419005f6d12b6c9ed5a35f9b59d50f1ae2595319987d19ba847418869853c',
    'events': '44258daf1a11fd15bcab90d341c2cf7276964f864022ed7d871ab6e2bf3b559c',
    'history': '44d1299f90cd78911926535228badbe5f1d18e6be29fd57b8f1f7382a0d0130b',
    'real_count_cv': '48e4c9a2873f470b7be10ff4e8a768b45fd7312cc31991237a68762842d6e9b7',
    'previous_config': 'e7b0645559c07b821e1cd200edf2c7889e6398592a91f7cc35d1291d0fa0174f',
}

                                                                           
N_MAIN_REPLICATES = int(os.environ.get('DEBRIS_ROBUST_MAIN_REPS', '200'))
N_SENSITIVITY_REPLICATES = int(os.environ.get('DEBRIS_ROBUST_SENS_REPS', '100'))
BUDGET_STEPS = int(os.environ.get('DEBRIS_ROBUST_BUDGET', '24'))
CHECKPOINT_EVERY = int(os.environ.get('DEBRIS_ROBUST_CHECKPOINT_EVERY', '10'))
SBC_DRAWS = int(os.environ.get('DEBRIS_ROBUST_SBC_DRAWS', '250'))
BOOTSTRAP_DRAWS = int(os.environ.get('DEBRIS_ROBUST_BOOTSTRAP_DRAWS', '5000'))
FORCE_RECOMPUTE = os.environ.get('DEBRIS_ROBUST_FORCE', '0') == '1'
RUN_CATALOG_AUDIT = os.environ.get('DEBRIS_RUN_CATALOG_AUDIT', '1') == '1'
RMSE_NONINFERIORITY_MARGIN_RELATIVE = 0.05

                                                                       
CAMPAIGN_START = pd.Timestamp('2018-01-04T12:00:00Z')
CAMPAIGN_END = pd.Timestamp('2018-01-05T12:00:00Z')
CATALOG_BLOCK_HOURS = 6
CATALOG_COARSE_STEP_SECONDS = 20.0
CATALOG_REFINE_STEP_SECONDS = 0.25
PROCESSED_RANGE_MIN_KM = 1026.639
PROCESSED_RANGE_MAX_KM = 1084.050
PRIMARY_BEAM_GATE_DEG = 5.0
BEAM_GATE_SENSITIVITY_DEG = (2.0, 3.5, 5.0)
SITE_LAT_DEG = 69.0 + 35.0 / 60.0 + 11.0 / 3600.0
SITE_LON_DEG = 19.0 + 13.0 / 60.0 + 38.0 / 3600.0
SITE_ALT_KM = 0.086
BEAM_AZIMUTH_DEG = 90.0
BEAM_ELEVATION_DEG = 75.0

MASTER_SEED = 20260829
FALSE_RATE_PER_HOUR = 0.973 / 24.0
DWELL_HOURS = 1.0
PRIMARY_RESPONSE_CORRELATION_DEG = 1.5
RESPONSE_CORRELATION_SENSITIVITY_DEG = (0.75, 3.0)
POLICIES = ('best_fixed', 'random', 'unweighted_aopt', 'eig', 'asset_aopt')
POLICY_LABELS = {
    'best_fixed': 'Best fixed',
    'random': 'Random',
    'unweighted_aopt': 'Unweighted adaptive',
    'eig': 'Information-gain adaptive',
    'asset_aopt': 'Satellite-weighted adaptive',
}
POLICY_COLORS = {
    'best_fixed': '#595959',
    'random': '#4C78A8',
    'unweighted_aopt': '#B279A2',
    'eig': '#54A24B',
    'asset_aopt': '#E45756',
}

print('research root:', PROJECT_ROOT)
print('Main replicates per scenario:', N_MAIN_REPLICATES)
print('Sensitivity replicates per setting:', N_SENSITIVITY_REPLICATES)
print('Observation budget:', BUDGET_STEPS, 'one-hour observations')


## 1. Frozen-input audit


In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


paths = {
    'response': RESPONSE_PATH,
    'response_uncertainty': RESPONSE_UNCERTAINTY_PATH,
    'assets': ASSET_PATH,
    'events': EVENT_PATH,
    'history': HISTORY_PATH,
    'real_count_cv': REAL_COUNT_CV_PATH,
    'previous_config': PREVIOUS_CONFIG_PATH,
}
missing = [str(path) for path in paths.values() if not path.is_file()]
if missing:
    raise FileNotFoundError('Required frozen inputs are missing:\n' + '\n'.join(missing))

actual_hashes = {name: sha256_file(path) for name, path in paths.items()}
hash_mismatches = {
    name: {'expected': EXPECTED_SHA256[name], 'actual': actual_hashes[name]}
    for name in EXPECTED_SHA256 if actual_hashes[name] != EXPECTED_SHA256[name]
}
if hash_mismatches:
    raise RuntimeError(f'Frozen-input checksum mismatch: {hash_mismatches}')

response_frame = pd.read_csv(RESPONSE_PATH)
uncertainty_frame = pd.read_csv(RESPONSE_UNCERTAINTY_PATH)
asset_frame = pd.read_csv(ASSET_PATH)
events = pd.read_csv(EVENT_PATH)
previous_config = json.loads(PREVIOUS_CONFIG_PATH.read_text())
real_count_cv = pd.read_csv(REAL_COUNT_CV_PATH)

shell_columns = [column for column in response_frame.columns if column.startswith('shell_')]
H = response_frame[shell_columns].to_numpy(dtype=float)
ELEVATIONS = response_frame.elevation_deg.to_numpy(dtype=float)
ALTITUDE_CENTERS = np.asarray([float(column.split('_')[1]) for column in shell_columns])
N_ACTIONS, N_LATENT = H.shape
ALTITUDE_EDGES = np.r_[ALTITUDE_CENTERS - 7.5, ALTITUDE_CENTERS[-1] + 7.5]

if H.shape != (49, 10) or not np.allclose(ELEVATIONS, np.arange(65.0, 89.0 + 0.01, 0.5)):
    raise RuntimeError(f'Expected a 49 x 10 exact response grid; found {H.shape}.')
if not np.isfinite(H).all() or (H < 0).any() or np.any(H.sum(axis=1) <= 0):
    raise RuntimeError('Invalid fine-grid response matrix.')

uncertainty_by_action = (
    uncertainty_frame.set_index('elevation_deg')
    .reindex(ELEVATIONS)
    .interpolate(method='index')
    .bfill()
    .ffill()
)
RESPONSE_RELATIVE_SD = uncertainty_by_action.relative_sigma.to_numpy(dtype=float)
RESPONSE_LOG_SD_BY_ACTION = np.sqrt(np.log1p(np.square(RESPONSE_RELATIVE_SD)))

asset_lookup = asset_frame.set_index('altitude_center_km')
ASSET_WEIGHTS = np.asarray(
    [asset_lookup.loc[altitude, 'active_asset_exposure_weight'] for altitude in ALTITUDE_CENTERS],
    dtype=float,
)
ASSET_WEIGHTS /= ASSET_WEIGHTS.sum()
EQUAL_WEIGHTS = np.repeat(1.0 / N_LATENT, N_LATENT)

if len(events) != 155 or events.association_tier.value_counts().to_dict() != {
    'unresolved': 106, 'tier_a_strict': 46, 'tier_b_empirical': 3,
}:
    raise RuntimeError('Frozen catalog event counts changed.')

print('INPUT AUDIT PASSED')
print('  Exact response actions / altitude shells:', H.shape)
print('  ORDEM action response relative SD range:',
      f'{RESPONSE_RELATIVE_SD.min():.3f} to {RESPONSE_RELATIVE_SD.max():.3f}')
print('  Radar events / strict / supported / unresolved:', 155, 46, 3, 106)
print('  Historical catalog file:', HISTORY_PATH.name)


## 2. Response-aware Bayesian model


In [ ]:
PRIOR_LOG_SD = float(previous_config['prior_log_sd'])
PRIOR_CORRELATION_LENGTH_KM = float(previous_config['prior_correlation_length_km'])
PRIOR_NUGGET_FRACTION = float(previous_config['prior_nugget_fraction'])
INFERENCE_DISPERSION = float(previous_config['negative_binomial_inference_dispersion'])
POINT_DISPERSION = float(previous_config['negative_binomial_point_dispersion'])
SIGNAL_RATE_CENTER_PER_HOUR = float(previous_config['signal_rate_center_per_hour'])
TRUTH_RESPONSE_LOG_SD = float(previous_config['response_perturbation_log_sd'])

shell_distance = np.abs(ALTITUDE_CENTERS[:, None] - ALTITUDE_CENTERS[None, :])
scaled_distance = np.sqrt(3.0) * shell_distance / PRIOR_CORRELATION_LENGTH_KM
matern32 = (1.0 + scaled_distance) * np.exp(-scaled_distance)
K_X = PRIOR_LOG_SD**2 * (
    (1.0 - PRIOR_NUGGET_FRACTION) * matern32
    + PRIOR_NUGGET_FRACTION * np.eye(N_LATENT)
)
K_X += np.eye(N_LATENT) * 1e-8
PRIOR_X_LOGDET = float(np.linalg.slogdet(K_X)[1])

CENTER_ACTION_INDEX = int(np.where(np.isclose(ELEVATIONS, 75.0))[0][0])
CENTER_RESPONSE_TOTAL = float(H[CENTER_ACTION_INDEX].sum())


@dataclass
class ModelContext:
    response_correlation_deg: float
    k_q: np.ndarray
    prior_covariance: np.ndarray
    prior_cholesky: np.ndarray
    best_fixed_action: int | None = None


def make_context(response_correlation_deg):
    action_distance = np.abs(ELEVATIONS[:, None] - ELEVATIONS[None, :])
    smooth = np.exp(-0.5 * np.square(action_distance / response_correlation_deg))
    response_correlation = 0.90 * smooth + 0.10 * np.eye(N_ACTIONS)
    k_q = (
        RESPONSE_LOG_SD_BY_ACTION[:, None]
        * response_correlation
        * RESPONSE_LOG_SD_BY_ACTION[None, :]
    )
    k_q += np.eye(N_ACTIONS) * 1e-8
    prior_covariance = np.zeros((N_LATENT + N_ACTIONS, N_LATENT + N_ACTIONS))
    prior_covariance[:N_LATENT, :N_LATENT] = K_X
    prior_covariance[N_LATENT:, N_LATENT:] = k_q
    return ModelContext(
        response_correlation_deg=float(response_correlation_deg),
        k_q=k_q,
        prior_covariance=prior_covariance,
        prior_cholesky=np.linalg.cholesky(prior_covariance),
    )


PRIMARY_CONTEXT = make_context(PRIMARY_RESPONSE_CORRELATION_DEG)
SENSITIVITY_CONTEXTS = {
    value: make_context(value) for value in RESPONSE_CORRELATION_SENSITIVITY_DEG
}
N_STATE = N_LATENT + N_ACTIONS
PRIOR_STATE_MEAN = np.zeros(N_STATE)


def stable_seed(*parts):
    payload = '|'.join(map(str, parts)).encode()
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], 'little') % (2**32 - 1)


def signal_scale(strength):
    return strength * SIGNAL_RATE_CENTER_PER_HOUR * DWELL_HOURS / CENTER_RESPONSE_TOTAL


def negative_binomial_draw(rng, mean, dispersion=INFERENCE_DISPERSION):
    mean = np.asarray(mean, dtype=float)
    probability = dispersion / (dispersion + mean)
    return rng.negative_binomial(dispersion, probability)


def observation_linearization(mean, covariance, action, strength, context):
    x = np.clip(mean[:N_LATENT], -4.0, 4.0)
    q = np.clip(mean[N_LATENT + action], -2.0, 2.0)
    mean_correction = 0.5 * context.k_q[action, action]
    components = (
        signal_scale(strength)
        * np.exp(q - mean_correction)
        * H[action]
        * np.exp(x)
    )
    signal = components.sum()
    expected = FALSE_RATE_PER_HOUR * DWELL_HOURS + signal
    gradient = np.zeros(N_STATE)
    gradient[:N_LATENT] = components
    gradient[N_LATENT + action] = signal
    observation_variance = expected + expected**2 / INFERENCE_DISPERSION
    return expected, gradient, observation_variance


def iterated_sequential_update(mean, covariance, action, observed, strength, context, iterations=5):
    prior_mean = mean.copy()
    prior_covariance = covariance.copy()
    trial = prior_mean.copy()
    for _ in range(iterations):
        expected, gradient, observation_variance = observation_linearization(
            trial, prior_covariance, action, strength, context
        )
        projected = prior_covariance @ gradient
        denominator = observation_variance + gradient @ projected
        adjusted_residual = observed - expected + gradient @ (trial - prior_mean)
        proposal = prior_mean + projected * adjusted_residual / max(denominator, 1e-10)
        proposal[:N_LATENT] = np.clip(proposal[:N_LATENT], -4.0, 4.0)
        proposal[N_LATENT:] = np.clip(proposal[N_LATENT:], -2.0, 2.0)
        trial = 0.35 * trial + 0.65 * proposal

    expected, gradient, observation_variance = observation_linearization(
        trial, prior_covariance, action, strength, context
    )
    projected = prior_covariance @ gradient
    denominator = observation_variance + gradient @ projected
    updated = prior_covariance - np.outer(projected, projected) / max(denominator, 1e-10)
    updated = 0.5 * (updated + updated.T)
    eigenvalues, eigenvectors = np.linalg.eigh(updated)
    updated = (eigenvectors * np.maximum(eigenvalues, 1e-10)) @ eigenvectors.T
    return trial, updated


def exact_batch_map_cov(history_actions, history_counts, strength, context, start_state):
    actions = np.asarray(history_actions, dtype=int)
    counts = np.asarray(history_counts, dtype=float)
    observed_h = H[actions]
    corrections = 0.5 * np.diag(context.k_q)[actions]
    cholesky = context.prior_cholesky

    def objective_gradient(z):
        state = cholesky @ z
        x = np.clip(state[:N_LATENT], -4.5, 4.5)
        q = np.clip(state[N_LATENT:][actions], -2.5, 2.5)
        response_scale = np.exp(q - corrections)
        components = (
            signal_scale(strength)
            * response_scale[:, None]
            * observed_h
            * np.exp(x)[None, :]
        )
        signal = components.sum(axis=1)
        means = np.maximum(FALSE_RATE_PER_HOUR * DWELL_HOURS + signal, 1e-10)
        k = INFERENCE_DISPERSION
        log_likelihood = (
            gammaln(counts + k) - gammaln(k) - gammaln(counts + 1.0)
            + k * np.log(k) + counts * np.log(means)
            - (counts + k) * np.log(means + k)
        )
        derivative_nll_mean = k * (means - counts) / (means * (means + k))
        gradient_state = np.zeros(N_STATE)
        gradient_state[:N_LATENT] = (
            derivative_nll_mean[:, None] * components
        ).sum(axis=0)
        np.add.at(
            gradient_state[N_LATENT:],
            actions,
            derivative_nll_mean * signal,
        )
        return float(0.5 * z @ z - log_likelihood.sum()), z + cholesky.T @ gradient_state

    initial_z = np.linalg.solve(cholesky, np.asarray(start_state, dtype=float))
    result = minimize(
        objective_gradient,
        initial_z,
        method='L-BFGS-B',
        jac=True,
        bounds=[(-7.0, 7.0)] * N_STATE,
        options={'maxiter': 180, 'maxls': 40, 'ftol': 1e-10, 'gtol': 2e-6},
    )
    state = cholesky @ np.asarray(result.x, dtype=float)
    x = np.clip(state[:N_LATENT], -4.5, 4.5)
    q = np.clip(state[N_LATENT:][actions], -2.5, 2.5)
    components = (
        signal_scale(strength)
        * np.exp(q - corrections)[:, None]
        * observed_h
        * np.exp(x)[None, :]
    )
    signal = components.sum(axis=1)
    means = np.maximum(FALSE_RATE_PER_HOUR * DWELL_HOURS + signal, 1e-10)
    gradient_state = np.zeros((len(actions), N_STATE))
    gradient_state[:, :N_LATENT] = components
    gradient_state[np.arange(len(actions)), N_LATENT + actions] = signal
    gradient_z = gradient_state @ cholesky
    observation_variance = means + means**2 / INFERENCE_DISPERSION
    precision_z = np.eye(N_STATE) + gradient_z.T @ (
        gradient_z / observation_variance[:, None]
    )
    covariance = cholesky @ np.linalg.inv(precision_z) @ cholesky.T
    covariance = 0.5 * (covariance + covariance.T)
    gradient_norm = float(np.linalg.norm(result.jac, ord=np.inf))
    converged = bool(result.success or (np.isfinite(result.fun) and gradient_norm < 1e-4))
    return state, covariance, converged, int(result.nit), gradient_norm


def theta_moments(mean, covariance):
    x_mean = mean[:N_LATENT]
    diagonal = np.maximum(np.diag(covariance)[:N_LATENT], 0.0)
    theta_mean = np.exp(x_mean + 0.5 * diagonal)
    theta_variance = (np.exp(diagonal) - 1.0) * np.exp(2.0 * x_mean + diagonal)
    return theta_mean, theta_variance


def interval_score(truth, low, high, alpha=0.05):
    score = high - low
    score += (2.0 / alpha) * (low - truth) * (truth < low)
    score += (2.0 / alpha) * (truth - high) * (truth > high)
    return score


def posterior_metrics(mean, covariance, truth_x, initial_asset_variance, initial_equal_variance):
    x_mean = mean[:N_LATENT]
    x_covariance = covariance[:N_LATENT, :N_LATENT]
    diagonal = np.maximum(np.diag(x_covariance), 0.0)
    sd = np.sqrt(diagonal)
    theta_mean, theta_variance = theta_moments(mean, covariance)
    truth_theta = np.exp(truth_x)
    squared_error = np.square(theta_mean - truth_theta)
    low95, high95 = np.exp(x_mean - 1.96 * sd), np.exp(x_mean + 1.96 * sd)
    low80, high80 = np.exp(x_mean - norm.ppf(0.90) * sd), np.exp(x_mean + norm.ppf(0.90) * sd)
    low50, high50 = np.exp(x_mean - norm.ppf(0.75) * sd), np.exp(x_mean + norm.ppf(0.75) * sd)
    sign, logdet = np.linalg.slogdet(x_covariance)
    return {
        'asset_weighted_posterior_variance': float(np.sum(ASSET_WEIGHTS * theta_variance)),
        'equal_weighted_posterior_variance': float(np.sum(EQUAL_WEIGHTS * theta_variance)),
        'asset_uncertainty_fraction': float(np.sum(ASSET_WEIGHTS * theta_variance) / initial_asset_variance),
        'equal_uncertainty_fraction': float(np.sum(EQUAL_WEIGHTS * theta_variance) / initial_equal_variance),
        'asset_weighted_rmse': float(np.sqrt(np.sum(ASSET_WEIGHTS * squared_error))),
        'equal_weighted_rmse': float(np.sqrt(np.sum(EQUAL_WEIGHTS * squared_error))),
        'asset_weighted_interval_score_95': float(
            np.sum(ASSET_WEIGHTS * interval_score(truth_theta, low95, high95))
        ),
        'cellwise_50_coverage': float(np.mean((truth_theta >= low50) & (truth_theta <= high50))),
        'cellwise_80_coverage': float(np.mean((truth_theta >= low80) & (truth_theta <= high80))),
        'cellwise_95_coverage': float(np.mean((truth_theta >= low95) & (truth_theta <= high95))),
        'asset_weighted_95_coverage': float(
            np.sum(ASSET_WEIGHTS * ((truth_theta >= low95) & (truth_theta <= high95)))
        ),
        'target_information_gain_nats': float(0.5 * (PRIOR_X_LOGDET - logdet)) if sign > 0 else np.nan,
    }


In [ ]:
def aopt_scores(mean, covariance, strength, context, weights):
    before = theta_moments(mean, covariance)[1]
    scores = np.empty(N_ACTIONS)
    x_mean = mean[:N_LATENT]
    for action in range(N_ACTIONS):
        _, gradient, observation_variance = observation_linearization(
            mean, covariance, action, strength, context
        )
        projected = covariance @ gradient
        denominator = observation_variance + gradient @ projected
        updated_diagonal = np.maximum(
            np.diag(covariance)[:N_LATENT]
            - np.square(projected[:N_LATENT]) / denominator,
            0.0,
        )
        after = (
            (np.exp(updated_diagonal) - 1.0)
            * np.exp(2.0 * x_mean + updated_diagonal)
        )
        scores[action] = np.sum(weights * (before - after))
    return scores


def eig_scores(mean, covariance, strength, context):
    scores = np.empty(N_ACTIONS)
    for action in range(N_ACTIONS):
        _, gradient, observation_variance = observation_linearization(
            mean, covariance, action, strength, context
        )
        ratio = gradient @ covariance @ gradient / observation_variance
        scores[action] = 0.5 * np.log1p(max(ratio, 0.0))
    return scores


def select_best_fixed(context, strength=1.0):
    predicted = []
    for action in range(N_ACTIONS):
        mean = PRIOR_STATE_MEAN.copy()
        covariance = context.prior_covariance.copy()
        for _ in range(BUDGET_STEPS):
            _, gradient, observation_variance = observation_linearization(
                mean, covariance, action, strength, context
            )
            projected = covariance @ gradient
            covariance -= np.outer(projected, projected) / (
                observation_variance + gradient @ projected
            )
            covariance = 0.5 * (covariance + covariance.T)
        predicted.append(np.sum(ASSET_WEIGHTS * theta_moments(mean, covariance)[1]))
    return int(np.argmin(predicted)), np.asarray(predicted)


for context in [PRIMARY_CONTEXT, *SENSITIVITY_CONTEXTS.values()]:
    context.best_fixed_action, context.fixed_design_variance = select_best_fixed(context)


def choose_action(policy, mean, covariance, strength, context, rng):
    if policy == 'best_fixed':
        return int(context.best_fixed_action), np.nan
    if policy == 'random':
        return int(rng.integers(N_ACTIONS)), np.nan
    if policy == 'unweighted_aopt':
        scores = aopt_scores(mean, covariance, strength, context, EQUAL_WEIGHTS)
        return int(np.argmax(scores)), float(scores.max())
    if policy == 'eig':
        scores = eig_scores(mean, covariance, strength, context)
        return int(np.argmax(scores)), float(scores.max())
    if policy == 'asset_aopt':
        scores = aopt_scores(mean, covariance, strength, context, ASSET_WEIGHTS)
        return int(np.argmax(scores)), float(scores.max())
    raise ValueError(policy)


print('RESPONSE-AWARE MODEL READY')
print('  Prior population log SD:', round(PRIOR_LOG_SD, 3))
print('  ORDEM action response log SD, median:', round(float(np.median(RESPONSE_LOG_SD_BY_ACTION)), 3))
print('  Primary response correlation length:', PRIMARY_RESPONSE_CORRELATION_DEG, 'degrees')
print('  Primary strong fixed elevation:', ELEVATIONS[PRIMARY_CONTEXT.best_fixed_action], 'degrees')
print('  Previous nominal ORDEM-flux fixed elevation:', previous_config['ordem_flux_fixed_elevation_deg'], 'degrees')
if np.isclose(
    ELEVATIONS[PRIMARY_CONTEXT.best_fixed_action],
    previous_config['ordem_flux_fixed_elevation_deg'],
):
    print('  Duplicate fixed baselines are intentionally combined in the new figures.')


## 3. Predeclared simulated worlds and policies


In [ ]:
def neighbor_mixing_matrix(fraction=0.15):
    matrix = np.zeros((N_LATENT, N_LATENT))
    for source in range(N_LATENT):
        neighbors = []
        if source > 0:
            neighbors.append(source - 1)
        if source < N_LATENT - 1:
            neighbors.append(source + 1)
        matrix[source, source] = 1.0 - fraction
        for destination in neighbors:
            matrix[destination, source] += fraction / len(neighbors)
    return matrix


SHELL_MIXING_MATRIX = neighbor_mixing_matrix(0.15)


def truth_and_streams(scenario, replicate, strength, experiment_block):
    rng = np.random.default_rng(stable_seed(
        MASTER_SEED, 'truth', experiment_block, scenario, strength, replicate
    ))
    base_x = rng.multivariate_normal(np.zeros(N_LATENT), K_X)
    truth_x = base_x.copy()
    truth_operator = H.copy()
    metadata = {
        'anomaly_center_km': np.nan,
        'mixing_fraction': 0.0,
        'truth_response_type': 'exact',
    }

    add_reported_response = scenario in {
        'reported_response_uncertainty', 'localized_anomaly', 'neighbor_shell_mixing'
    }
    if add_reported_response:
        truth_q = rng.multivariate_normal(np.zeros(N_ACTIONS), PRIMARY_CONTEXT.k_q)
        truth_operator = H * np.exp(
            truth_q[:, None] - 0.5 * np.diag(PRIMARY_CONTEXT.k_q)[:, None]
        )
        metadata['truth_response_type'] = 'reported_action_uncertainty'

    if scenario == 'localized_anomaly':
        center_index = int(rng.integers(N_LATENT))
        width_km = float(rng.uniform(8.0, 15.0))
        amplitude = float(rng.uniform(np.log(2.0), np.log(3.0)))
        bump = amplitude * np.exp(
            -0.5 * np.square((ALTITUDE_CENTERS - ALTITUDE_CENTERS[center_index]) / width_km)
        )
        truth_x = base_x + bump
        metadata['anomaly_center_km'] = float(ALTITUDE_CENTERS[center_index])
    elif scenario == 'neighbor_shell_mixing':
        truth_theta = SHELL_MIXING_MATRIX @ np.exp(base_x)
        truth_x = np.log(np.maximum(truth_theta, 1e-12))
        metadata['mixing_fraction'] = 0.15
    elif scenario == 'full_cell_response_perturbation':
        raw = rng.normal(size=H.shape)
        smooth = gaussian_filter(raw, sigma=(2.0, 1.0), mode='reflect')
        smooth = (smooth - smooth.mean()) / max(smooth.std(ddof=1), 1e-12)
        truth_operator = H * np.exp(
            TRUTH_RESPONSE_LOG_SD * smooth - 0.5 * TRUTH_RESPONSE_LOG_SD**2
        )
        metadata['truth_response_type'] = 'smooth_action_shell_perturbation'
    elif scenario not in {'nominal_response', 'reported_response_uncertainty'}:
        raise ValueError(scenario)

    truth_x = np.clip(truth_x, -3.0, 3.0)
    means = (
        FALSE_RATE_PER_HOUR * DWELL_HOURS
        + signal_scale(strength) * (truth_operator @ np.exp(truth_x))
    )
    count_rng = np.random.default_rng(stable_seed(
        MASTER_SEED, 'counts', experiment_block, scenario, strength, replicate
    ))
    probability = INFERENCE_DISPERSION / (
        INFERENCE_DISPERSION + means[:, None]
    )
    streams = count_rng.negative_binomial(
        INFERENCE_DISPERSION,
        probability,
        size=(N_ACTIONS, BUDGET_STEPS),
    )
    return truth_x, streams, means, metadata


def shell_rows_from_state(
    final_mean, final_covariance, truth_x, condition, replicate, policy
):
    x_mean = final_mean[:N_LATENT]
    diagonal = np.maximum(np.diag(final_covariance)[:N_LATENT], 0.0)
    sd = np.sqrt(diagonal)
    theta_mean = np.exp(x_mean + 0.5 * diagonal)
    theta_variance = (np.exp(diagonal) - 1.0) * np.exp(2.0 * x_mean + diagonal)
    truth_theta = np.exp(truth_x)
    low95 = np.exp(x_mean - 1.96 * sd)
    high95 = np.exp(x_mean + 1.96 * sd)
    rows = []
    for index, altitude in enumerate(ALTITUDE_CENTERS):
        rows.append({
            **condition,
            'replicate': replicate,
            'policy': policy,
            'latent_index': index,
            'altitude_center_km': altitude,
            'asset_weight': ASSET_WEIGHTS[index],
            'truth_multiplier': truth_theta[index],
            'posterior_multiplier_mean': theta_mean[index],
            'posterior_multiplier_variance': theta_variance[index],
            'posterior_log_mean': x_mean[index],
            'posterior_log_sd': sd[index],
            'covered_95': bool(low95[index] <= truth_theta[index] <= high95[index]),
        })
    return rows


def aggregate_sbc_rows(final_mean, final_covariance, truth_x, condition, replicate, policy):
    rows = []
    x_covariance = final_covariance[:N_LATENT, :N_LATENT]
    for aggregate_name, weights in (
        ('asset_weighted_log_multiplier', ASSET_WEIGHTS),
        ('equal_weighted_log_multiplier', EQUAL_WEIGHTS),
    ):
        posterior_mean = float(weights @ final_mean[:N_LATENT])
        posterior_sd = float(np.sqrt(max(weights @ x_covariance @ weights, 1e-12)))
        truth = float(weights @ truth_x)
        rng = np.random.default_rng(stable_seed(
            MASTER_SEED, 'aggregate_sbc', aggregate_name,
            condition['experiment_block'], condition['scenario'],
            condition['strength'], condition['response_correlation_deg'], replicate,
        ))
        draws = rng.normal(posterior_mean, posterior_sd, size=SBC_DRAWS)
        rows.append({
            **condition,
            'replicate': replicate,
            'policy': policy,
            'aggregate': aggregate_name,
            'truth': truth,
            'posterior_mean': posterior_mean,
            'posterior_sd': posterior_sd,
            'rank': int(np.sum(draws < truth)),
            'draws': SBC_DRAWS,
        })
    return rows


def run_policy(policy, truth_x, streams, strength, condition, replicate, context):
    policy_rng = np.random.default_rng(stable_seed(
        MASTER_SEED, 'policy', policy,
        condition['experiment_block'], condition['scenario'],
        strength, condition['response_correlation_deg'], replicate,
    ))
    mean = PRIOR_STATE_MEAN.copy()
    covariance = context.prior_covariance.copy()
    initial_theta_variance = theta_moments(mean, covariance)[1]
    initial_asset_variance = float(np.sum(ASSET_WEIGHTS * initial_theta_variance))
    initial_equal_variance = float(np.sum(EQUAL_WEIGHTS * initial_theta_variance))
    visits = np.zeros(N_ACTIONS, dtype=int)
    history_actions, history_counts = [], []
    step_rows, action_rows = [], []

    initial = posterior_metrics(
        mean, covariance, truth_x, initial_asset_variance, initial_equal_variance
    )
    initial.update({
        **condition, 'replicate': replicate, 'policy': policy, 'step': 0,
        'cumulative_events': 0, 'posterior_method': 'prior',
    })
    step_rows.append(initial)

    for step in range(1, BUDGET_STEPS + 1):
        selected, decision_score = choose_action(
            policy, mean, covariance, strength, context, policy_rng
        )
        visit_index = int(visits[selected])
        observed = int(streams[selected, visit_index])
        visits[selected] += 1
        predicted_before = observation_linearization(
            mean, covariance, selected, strength, context
        )[0]
        history_actions.append(selected)
        history_counts.append(observed)
        mean, covariance = iterated_sequential_update(
            mean, covariance, selected, observed, strength, context
        )
        metrics = posterior_metrics(
            mean, covariance, truth_x, initial_asset_variance, initial_equal_variance
        )
        metrics.update({
            **condition, 'replicate': replicate, 'policy': policy, 'step': step,
            'cumulative_events': int(sum(history_counts)),
            'posterior_method': 'iterated_sequential',
        })
        step_rows.append(metrics)
        action_rows.append({
            **condition, 'replicate': replicate, 'policy': policy, 'step': step,
            'action_index': selected,
            'elevation_deg': ELEVATIONS[selected],
            'visit_number': visit_index + 1,
            'observed_events': observed,
            'predicted_events_before_update': predicted_before,
            'decision_score': decision_score,
        })

    filter_final_mean = mean.copy()
    filter_final_covariance = covariance.copy()
    exact_mean, exact_covariance, optimizer_success, optimizer_iterations, gradient_norm = (
        exact_batch_map_cov(
            history_actions, history_counts, strength, context, filter_final_mean
        )
    )
    exact_metrics = posterior_metrics(
        exact_mean, exact_covariance, truth_x,
        initial_asset_variance, initial_equal_variance,
    )
    step_rows[-1].update(exact_metrics)
    step_rows[-1]['posterior_method'] = 'exact_nb_batch_final'
    step_rows[-1]['optimizer_success'] = optimizer_success
    step_rows[-1]['optimizer_iterations'] = optimizer_iterations
    step_rows[-1]['optimizer_gradient_inf_norm'] = gradient_norm
    step_rows[-1]['filter_exact_asset_uncertainty_relative_difference'] = (
        abs(
            posterior_metrics(
                filter_final_mean, filter_final_covariance, truth_x,
                initial_asset_variance, initial_equal_variance,
            )['asset_uncertainty_fraction']
            - exact_metrics['asset_uncertainty_fraction']
        ) / max(exact_metrics['asset_uncertainty_fraction'], 1e-12)
    )

    shells = shell_rows_from_state(
        exact_mean, exact_covariance, truth_x, condition, replicate, policy
    )
    sbc = aggregate_sbc_rows(
        exact_mean, exact_covariance, truth_x, condition, replicate, policy
    ) if policy == 'asset_aopt' else []
    return step_rows, action_rows, shells, sbc


In [ ]:
main_scenarios = (
    'nominal_response',
    'reported_response_uncertainty',
    'localized_anomaly',
    'neighbor_shell_mixing',
    'full_cell_response_perturbation',
)
experiment_conditions = [
    {
        'experiment_block': 'main',
        'scenario': scenario,
        'strength': 1.0,
        'response_correlation_deg': PRIMARY_RESPONSE_CORRELATION_DEG,
        'n_replicates': N_MAIN_REPLICATES,
    }
    for scenario in main_scenarios
]
for strength in (0.5, 2.0):
    experiment_conditions.append({
        'experiment_block': 'strength_sensitivity',
        'scenario': 'reported_response_uncertainty',
        'strength': strength,
        'response_correlation_deg': PRIMARY_RESPONSE_CORRELATION_DEG,
        'n_replicates': N_SENSITIVITY_REPLICATES,
    })
for correlation_length in RESPONSE_CORRELATION_SENSITIVITY_DEG:
    experiment_conditions.append({
        'experiment_block': 'correlation_sensitivity',
        'scenario': 'full_cell_response_perturbation',
        'strength': 1.0,
        'response_correlation_deg': correlation_length,
        'n_replicates': N_SENSITIVITY_REPLICATES,
    })

experiment_config = {
    'model_version': 'debris_response_aware_nb_joint_laplace_v2',
    'created_for': 'publication_lock_validation',
    'primary_policy': 'asset_aopt',
    'primary_target': 'active-satellite-exposure-weighted population uncertainty',
    'actions': int(N_ACTIONS),
    'altitude_shells': int(N_LATENT),
    'budget_steps': BUDGET_STEPS,
    'policies': list(POLICIES),
    'main_scenarios': list(main_scenarios),
    'conditions': experiment_conditions,
    'primary_response_correlation_deg': PRIMARY_RESPONSE_CORRELATION_DEG,
    'response_correlation_sensitivity_deg': list(RESPONSE_CORRELATION_SENSITIVITY_DEG),
    'response_uncertainty_source': 'ORDEM action-level range-plus-population sigma',
    'response_uncertainty_is_systematic_across_hours': True,
    'sequential_action_posterior': 'iterated Gaussian/NB moment update',
    'final_metric_posterior': 'joint negative-binomial MAP with Fisher-Laplace covariance',
    'main_replicates': N_MAIN_REPLICATES,
    'sensitivity_replicates': N_SENSITIVITY_REPLICATES,
    'sbc_draws': SBC_DRAWS,
    'master_seed': MASTER_SEED,
    'input_hashes': actual_hashes,
    'catalog_passage_audit': {
        'campaign_start_utc': CAMPAIGN_START.isoformat(),
        'campaign_end_utc': CAMPAIGN_END.isoformat(),
        'historical_tle_policy': 'latest TLE at or before each six-hour block start',
        'block_hours': CATALOG_BLOCK_HOURS,
        'coarse_step_seconds': CATALOG_COARSE_STEP_SECONDS,
        'refine_step_seconds': CATALOG_REFINE_STEP_SECONDS,
        'processed_range_km': [PROCESSED_RANGE_MIN_KM, PROCESSED_RANGE_MAX_KM],
        'primary_beam_gate_deg': PRIMARY_BEAM_GATE_DEG,
        'site_latitude_deg': SITE_LAT_DEG,
        'site_longitude_deg': SITE_LON_DEG,
        'site_altitude_km': SITE_ALT_KM,
        'beam_azimuth_deg': BEAM_AZIMUTH_DEG,
        'beam_elevation_deg': BEAM_ELEVATION_DEG,
    },
}
config_hash = hashlib.sha256(
    json.dumps(experiment_config, sort_keys=True).encode()
).hexdigest()[:12]
                                                                              
                                                                             
experiment_config['claim_analysis_thresholds'] = {
    'hard_response_rmse_noninferiority_margin_relative_to_random_mean': (
        RMSE_NONINFERIORITY_MARGIN_RELATIVE
    ),
    'confidence_interval': 0.95,
    'paired_unit': 'complete simulated world',
}
OUTPUT_DIR = (
    PROJECT_ROOT / 'phase3_outputs' / 'publication_lock_validation'
    / f'robust_lock_{config_hash}'
)
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'publication_lock_config.json').write_text(
    json.dumps(experiment_config, indent=2)
)


def context_for_condition(condition):
    value = float(condition['response_correlation_deg'])
    if np.isclose(value, PRIMARY_RESPONSE_CORRELATION_DEG):
        return PRIMARY_CONTEXT
    return SENSITIVITY_CONTEXTS[value]


def condition_token(condition):
    return (
        f"{condition['experiment_block']}_{condition['scenario']}"
        f"_s{condition['strength']:g}_corr{condition['response_correlation_deg']:g}"
    ).replace('.', 'p')


print('OUTPUT DIRECTORY:', OUTPUT_DIR)
print('Configuration hash:', config_hash)
print('Predeclared condition groups:', len(experiment_conditions))


## 4. Run or resume the response-aware experiment


In [ ]:
all_step_frames = []
all_action_frames = []
all_shell_frames = []
all_sbc_frames = []
run_started = time.time()

for condition_number, condition_spec in enumerate(experiment_conditions, start=1):
    condition = {
        key: value for key, value in condition_spec.items() if key != 'n_replicates'
    }
    n_replicates = int(condition_spec['n_replicates'])
    token = condition_token(condition_spec)
    paths_for_condition = {
        'steps': CHECKPOINT_DIR / f'steps_{token}.csv.gz',
        'actions': CHECKPOINT_DIR / f'actions_{token}.csv.gz',
        'shells': CHECKPOINT_DIR / f'shells_{token}.csv.gz',
        'sbc': CHECKPOINT_DIR / f'sbc_{token}.csv.gz',
    }
    if FORCE_RECOMPUTE:
        for path in paths_for_condition.values():
            if path.exists():
                path.unlink()

    existing_steps = (
        pd.read_csv(paths_for_condition['steps'])
        if paths_for_condition['steps'].exists() else pd.DataFrame()
    )
    existing_actions = (
        pd.read_csv(paths_for_condition['actions'])
        if paths_for_condition['actions'].exists() else pd.DataFrame()
    )
    existing_shells = (
        pd.read_csv(paths_for_condition['shells'])
        if paths_for_condition['shells'].exists() else pd.DataFrame()
    )
    existing_sbc = (
        pd.read_csv(paths_for_condition['sbc'])
        if paths_for_condition['sbc'].exists() else pd.DataFrame()
    )
    if len(existing_actions):
        counts = existing_actions.groupby(['replicate', 'policy']).size()
        completed = set(
            counts[counts == BUDGET_STEPS]
            .reset_index()
            .groupby('replicate')
            .filter(lambda group: set(group.policy) == set(POLICIES))
            .replicate.astype(int)
        )
    else:
        completed = set()

    print(
        f"\nCondition {condition_number}/{len(experiment_conditions)}: "
        f"{condition['experiment_block']} / {condition['scenario']} / "
        f"strength {condition['strength']:g} / corr {condition['response_correlation_deg']:g}°"
    )
    print('  Resuming completed worlds:', len(completed), '/', n_replicates)
    context = context_for_condition(condition_spec)
    pending_steps, pending_actions, pending_shells, pending_sbc = [], [], [], []

    def flush_pending():
        nonlocal_placeholder = None
        frames_and_paths = (
            (pending_steps, existing_steps, paths_for_condition['steps']),
            (pending_actions, existing_actions, paths_for_condition['actions']),
            (pending_shells, existing_shells, paths_for_condition['shells']),
            (pending_sbc, existing_sbc, paths_for_condition['sbc']),
        )
        updated = []
        for pending, existing, path in frames_and_paths:
            if pending:
                existing = pd.concat([existing, pd.DataFrame(pending)], ignore_index=True)
                existing.to_csv(path, index=False, compression='gzip')
            updated.append(existing)
        return updated

    processed_since_flush = 0
    for replicate in range(n_replicates):
        if replicate in completed:
            continue
        truth_x, streams, truth_means, metadata = truth_and_streams(
            condition['scenario'], replicate, float(condition['strength']),
            condition['experiment_block'],
        )
        for policy in POLICIES:
            step_rows, action_rows, shell_rows, sbc_rows = run_policy(
                policy, truth_x, streams, float(condition['strength']),
                condition, replicate, context,
            )
            for row in step_rows:
                row.update(metadata)
            for row in action_rows:
                row.update(metadata)
            for row in shell_rows:
                row.update(metadata)
            for row in sbc_rows:
                row.update(metadata)
            pending_steps.extend(step_rows)
            pending_actions.extend(action_rows)
            pending_shells.extend(shell_rows)
            pending_sbc.extend(sbc_rows)
        processed_since_flush += 1

        if processed_since_flush >= CHECKPOINT_EVERY:
            existing_steps, existing_actions, existing_shells, existing_sbc = flush_pending()
            pending_steps.clear(); pending_actions.clear(); pending_shells.clear(); pending_sbc.clear()
            processed_since_flush = 0
            print(f'  checkpoint through replicate {replicate + 1}/{n_replicates}', flush=True)

    if pending_steps or pending_actions or pending_shells or pending_sbc:
        existing_steps, existing_actions, existing_shells, existing_sbc = flush_pending()
        pending_steps.clear(); pending_actions.clear(); pending_shells.clear(); pending_sbc.clear()

    expected_action_rows = n_replicates * len(POLICIES) * BUDGET_STEPS
    if len(existing_actions) != expected_action_rows:
        raise RuntimeError(
            f'{token}: action rows {len(existing_actions)} != {expected_action_rows}'
        )
    all_step_frames.append(existing_steps)
    all_action_frames.append(existing_actions)
    all_shell_frames.append(existing_shells)
    if len(existing_sbc):
        all_sbc_frames.append(existing_sbc)

step_metrics = pd.concat(all_step_frames, ignore_index=True)
action_history = pd.concat(all_action_frames, ignore_index=True)
shell_metrics = pd.concat(all_shell_frames, ignore_index=True)
aggregate_sbc = pd.concat(all_sbc_frames, ignore_index=True)

print('\nRESPONSE-AWARE SIMULATION COMPLETE')
print('  Step rows:', f'{len(step_metrics):,}')
print('  Action rows:', f'{len(action_history):,}')
print('  Shell rows:', f'{len(shell_metrics):,}')
print('  Aggregate SBC rows:', f'{len(aggregate_sbc):,}')
print('  Simulation wall time, minutes:', round((time.time() - run_started) / 60.0, 2))


## 5. Paired inference, calibration, and weighting diagnostics


In [ ]:
condition_columns = [
    'experiment_block', 'scenario', 'strength', 'response_correlation_deg'
]
final_metrics = step_metrics.loc[step_metrics.step == BUDGET_STEPS].copy()


def mean_ci(values):
    values = np.asarray(values, dtype=float)
    mean = float(np.mean(values))
    if len(values) <= 1:
        return mean, np.nan, np.nan
    half = 1.96 * float(np.std(values, ddof=1)) / np.sqrt(len(values))
    return mean, mean - half, mean + half


def bootstrap_mean_ci(values, seed, draws=BOOTSTRAP_DRAWS):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    means = np.empty(draws)
    for index in range(draws):
        means[index] = rng.choice(values, len(values), replace=True).mean()
    return float(values.mean()), *np.quantile(means, [0.025, 0.975]).tolist()


curve_metric_names = [
    'asset_uncertainty_fraction', 'asset_weighted_rmse',
    'asset_weighted_interval_score_95', 'cellwise_95_coverage',
]
curve_rows = []
for keys, group in step_metrics.groupby(condition_columns + ['policy', 'step']):
    row = dict(zip(condition_columns + ['policy', 'step'], keys))
    row['n'] = group.replicate.nunique()
    for metric in curve_metric_names:
        mean, low, high = mean_ci(group[metric])
        row[f'{metric}_mean'] = mean
        row[f'{metric}_ci95_low'] = low
        row[f'{metric}_ci95_high'] = high
    curve_rows.append(row)
curve_summary = pd.DataFrame(curve_rows)

final_metric_names = [
    'asset_uncertainty_fraction', 'equal_uncertainty_fraction',
    'asset_weighted_rmse', 'equal_weighted_rmse',
    'asset_weighted_interval_score_95',
    'cellwise_50_coverage', 'cellwise_80_coverage', 'cellwise_95_coverage',
    'asset_weighted_95_coverage', 'target_information_gain_nats',
]
summary_rows = []
for keys, group in final_metrics.groupby(condition_columns + ['policy']):
    row = dict(zip(condition_columns + ['policy'], keys))
    row['n'] = group.replicate.nunique()
    optimizer_success = (
        group.optimizer_success.astype(str).str.lower().eq('true')
    )
    row['optimizer_failure_rate'] = float(1.0 - optimizer_success.mean())
    row['filter_exact_relative_difference_median'] = float(
        group.filter_exact_asset_uncertainty_relative_difference.median()
    )
    for metric in final_metric_names:
        mean, low, high = bootstrap_mean_ci(
            group[metric],
            stable_seed(MASTER_SEED, 'summary_bootstrap', *keys, metric),
            draws=min(BOOTSTRAP_DRAWS, 3000),
        )
        row[f'{metric}_mean'] = mean
        row[f'{metric}_ci95_low'] = low
        row[f'{metric}_ci95_high'] = high
    summary_rows.append(row)
final_summary = pd.DataFrame(summary_rows)

pairwise_rows = []
for condition_keys, condition_group in final_metrics.groupby(condition_columns):
    proposed = condition_group.loc[condition_group.policy == 'asset_aopt'].set_index('replicate')
    for baseline in [policy for policy in POLICIES if policy != 'asset_aopt']:
        comparator = condition_group.loc[condition_group.policy == baseline].set_index('replicate')
        shared = proposed.index.intersection(comparator.index)
        row = dict(zip(condition_columns, condition_keys))
        row['baseline'] = baseline
        row['paired_replicates'] = len(shared)
        for metric, short_name in (
            ('asset_uncertainty_fraction', 'uncertainty'),
            ('asset_weighted_rmse', 'rmse'),
            ('asset_weighted_interval_score_95', 'interval_score'),
        ):
            proposed_values = proposed.loc[shared, metric].to_numpy()
            baseline_values = comparator.loc[shared, metric].to_numpy()
            improvement = 100.0 * (baseline_values - proposed_values) / np.maximum(
                baseline_values, 1e-12
            )
            mean, low, high = bootstrap_mean_ci(
                improvement,
                stable_seed(MASTER_SEED, 'paired_bootstrap', *condition_keys, baseline, metric),
            )
            row[f'{short_name}_improvement_percent_mean'] = mean
            row[f'{short_name}_improvement_ci95_low'] = low
            row[f'{short_name}_improvement_ci95_high'] = high
            row[f'{short_name}_win_rate'] = float(np.mean(improvement > 0))
            raw_difference = baseline_values - proposed_values
            raw_mean, raw_low, raw_high = bootstrap_mean_ci(
                raw_difference,
                stable_seed(
                    MASTER_SEED, 'paired_raw_bootstrap',
                    *condition_keys, baseline, metric,
                ),
            )
            row[f'{short_name}_raw_difference_mean'] = raw_mean
            row[f'{short_name}_raw_difference_ci95_low'] = raw_low
            row[f'{short_name}_raw_difference_ci95_high'] = raw_high
            ratio_rng = np.random.default_rng(stable_seed(
                MASTER_SEED, 'paired_ratio_of_means_bootstrap',
                *condition_keys, baseline, metric,
            ))
            ratio_draws = []
            for _ in range(BOOTSTRAP_DRAWS):
                sample = ratio_rng.integers(0, len(shared), len(shared))
                baseline_mean = baseline_values[sample].mean()
                proposed_mean = proposed_values[sample].mean()
                ratio_draws.append(
                    100.0 * (baseline_mean - proposed_mean)
                    / max(baseline_mean, 1e-12)
                )
            row[f'{short_name}_ratio_of_means_improvement_percent'] = float(
                100.0 * (baseline_values.mean() - proposed_values.mean())
                / max(baseline_values.mean(), 1e-12)
            )
            row[f'{short_name}_ratio_of_means_ci95_low'] = float(
                np.quantile(ratio_draws, 0.025)
            )
            row[f'{short_name}_ratio_of_means_ci95_high'] = float(
                np.quantile(ratio_draws, 0.975)
            )
        pairwise_rows.append(row)
pairwise = pd.DataFrame(pairwise_rows)

coverage_rows = []
for keys, group in final_metrics.groupby(condition_columns + ['policy']):
    row = dict(zip(condition_columns + ['policy'], keys))
    for level, column in (
        (0.50, 'cellwise_50_coverage'),
        (0.80, 'cellwise_80_coverage'),
        (0.95, 'cellwise_95_coverage'),
    ):
        mean, low, high = bootstrap_mean_ci(
            group[column],
            stable_seed(MASTER_SEED, 'coverage_bootstrap', *keys, level),
            draws=min(BOOTSTRAP_DRAWS, 3000),
        )
        coverage_rows.append({
            **row, 'nominal_level': level, 'observed_coverage': mean,
            'ci95_low': low, 'ci95_high': high,
        })
coverage_reliability = pd.DataFrame(coverage_rows)

shell_ablation_rows = []
main_shells = shell_metrics.loc[
    (shell_metrics.experiment_block == 'main')
    & (shell_metrics.strength == 1.0)
    & np.isclose(shell_metrics.response_correlation_deg, PRIMARY_RESPONSE_CORRELATION_DEG)
]
for scenario, scenario_group in main_shells.groupby('scenario'):
    proposed = scenario_group.loc[scenario_group.policy == 'asset_aopt'].set_index(
        ['replicate', 'latent_index']
    )
    unweighted = scenario_group.loc[
        scenario_group.policy == 'unweighted_aopt'
    ].set_index(['replicate', 'latent_index'])
    shared = proposed.index.intersection(unweighted.index)
    joined = pd.DataFrame({
        'proposed': proposed.loc[shared, 'posterior_multiplier_variance'],
        'unweighted': unweighted.loc[shared, 'posterior_multiplier_variance'],
        'altitude': proposed.loc[shared, 'altitude_center_km'],
        'asset_weight': proposed.loc[shared, 'asset_weight'],
    }).reset_index()
    joined['improvement'] = 100.0 * (
        joined.unweighted - joined.proposed
    ) / np.maximum(joined.unweighted, 1e-12)
    for altitude, shell_group in joined.groupby('altitude'):
        mean, low, high = bootstrap_mean_ci(
            shell_group.improvement,
            stable_seed(MASTER_SEED, 'shell_ablation', scenario, altitude),
            draws=min(BOOTSTRAP_DRAWS, 3000),
        )
        shell_ablation_rows.append({
            'scenario': scenario,
            'altitude_center_km': altitude,
            'asset_weight': float(shell_group.asset_weight.iloc[0]),
            'asset_vs_unweighted_variance_improvement_percent_mean': mean,
            'ci95_low': low,
            'ci95_high': high,
        })
shell_ablation = pd.DataFrame(shell_ablation_rows)

                                                                        
aggregate_sbc_summary_rows = []
rank_bins = np.linspace(0, SBC_DRAWS + 1, 11)
for keys, group in aggregate_sbc.groupby(condition_columns + ['aggregate']):
    counts, _ = np.histogram(group['rank'].to_numpy(), bins=rank_bins)
    statistic, p_value = chisquare(counts)
    aggregate_sbc_summary_rows.append({
        **dict(zip(condition_columns + ['aggregate'], keys)),
        'independent_worlds': group.replicate.nunique(),
        'rank_decile_counts': json.dumps(counts.tolist()),
        'chi_square': float(statistic),
        'p_value': float(p_value),
    })
aggregate_sbc_summary = pd.DataFrame(aggregate_sbc_summary_rows)

                                                                       
real_count_cv = real_count_cv.copy()
real_count_cv['nb_minus_poisson_log_score'] = (
    real_count_cv.nb_log_score - real_count_cv.poisson_log_score
)
fold_deltas = real_count_cv.groupby('fold').nb_minus_poisson_log_score.mean().to_numpy()
bootstrap_rng = np.random.default_rng(stable_seed(MASTER_SEED, 'real_count_block_bootstrap'))
bootstrap_deltas = np.asarray([
    bootstrap_rng.choice(fold_deltas, len(fold_deltas), replace=True).mean()
    for _ in range(max(BOOTSTRAP_DRAWS, 10000))
])
real_count_model_comparison = {
    'independent_four_hour_blocks': int(len(fold_deltas)),
    'nb_minus_poisson_mean_log_score': float(real_count_cv.nb_minus_poisson_log_score.mean()),
    'block_bootstrap_ci95_low': float(np.quantile(bootstrap_deltas, 0.025)),
    'block_bootstrap_ci95_high': float(np.quantile(bootstrap_deltas, 0.975)),
    'nb_80_coverage': float(real_count_cv.nb_80_covered.mean()),
    'nb_95_coverage': float(real_count_cv.nb_95_covered.mean()),
    'poisson_80_coverage': float(real_count_cv.poisson_80_covered.mean()),
    'poisson_95_coverage': float(real_count_cv.poisson_95_covered.mean()),
    'interpretation': (
        'The NB model is retained for conservative overdispersion. Its log-score advantage '
        'is called clear only if the block-bootstrap interval excludes zero.'
    ),
}

print('STATISTICAL SUMMARIES COMPLETE')
display(final_summary.loc[
    (final_summary.experiment_block == 'main')
    & (final_summary.policy.isin(['best_fixed', 'random', 'unweighted_aopt', 'asset_aopt']))
].sort_values(['scenario', 'policy']))
print('Real-count block-bootstrap comparison:')
display(pd.DataFrame([real_count_model_comparison]))


## 6. Catalog-first passage recovery audit


In [ ]:
def datetime_jd(times):
    unix_seconds = times.view('int64') / 1e9
    full = unix_seconds / 86400.0 + 2440587.5
    whole = np.floor(full)
    return whole, full - whole


def make_site_geometry():
    latitude = np.deg2rad(SITE_LAT_DEG)
    longitude = np.deg2rad(SITE_LON_DEG)
    semi_major = 6378.137
    flattening = 1.0 / 298.257223563
    eccentricity_squared = flattening * (2.0 - flattening)
    prime_vertical = semi_major / np.sqrt(
        1.0 - eccentricity_squared * np.sin(latitude) ** 2
    )
    site = np.asarray([
        (prime_vertical + SITE_ALT_KM) * np.cos(latitude) * np.cos(longitude),
        (prime_vertical + SITE_ALT_KM) * np.cos(latitude) * np.sin(longitude),
        (prime_vertical * (1.0 - eccentricity_squared) + SITE_ALT_KM)
        * np.sin(latitude),
    ])
    azimuth = np.deg2rad(BEAM_AZIMUTH_DEG)
    elevation = np.deg2rad(BEAM_ELEVATION_DEG)
    beam_enu = np.asarray([
        np.cos(elevation) * np.sin(azimuth),
        np.cos(elevation) * np.cos(azimuth),
        np.sin(elevation),
    ])
    return latitude, longitude, site, beam_enu


SITE_LAT_RAD, SITE_LON_RAD, SITE_ECEF_KM, BEAM_ENU = make_site_geometry()


def look_from_teme(position_teme_km, full_julian_date):
    centuries = (full_julian_date - 2451545.0) / 36525.0
    gmst = np.deg2rad((
        280.46061837
        + 360.98564736629 * (full_julian_date - 2451545.0)
        + 0.000387933 * centuries**2
        - centuries**3 / 38710000.0
    ) % 360.0)
    cosine, sine = np.cos(gmst), np.sin(gmst)
    x_teme = position_teme_km[..., 0]
    y_teme = position_teme_km[..., 1]
    z_teme = position_teme_km[..., 2]
    x_ecef = cosine * x_teme + sine * y_teme
    y_ecef = -sine * x_teme + cosine * y_teme
    dx = x_ecef - SITE_ECEF_KM[0]
    dy = y_ecef - SITE_ECEF_KM[1]
    dz = z_teme - SITE_ECEF_KM[2]
    east = -np.sin(SITE_LON_RAD) * dx + np.cos(SITE_LON_RAD) * dy
    north = (
        -np.sin(SITE_LAT_RAD) * np.cos(SITE_LON_RAD) * dx
        - np.sin(SITE_LAT_RAD) * np.sin(SITE_LON_RAD) * dy
        + np.cos(SITE_LAT_RAD) * dz
    )
    up = (
        np.cos(SITE_LAT_RAD) * np.cos(SITE_LON_RAD) * dx
        + np.cos(SITE_LAT_RAD) * np.sin(SITE_LON_RAD) * dy
        + np.sin(SITE_LAT_RAD) * dz
    )
    slant_range = np.sqrt(east**2 + north**2 + up**2)
    beam_dot = (
        east * BEAM_ENU[0] + north * BEAM_ENU[1] + up * BEAM_ENU[2]
    ) / slant_range
    beam_separation = np.rad2deg(np.arccos(np.clip(beam_dot, -1.0, 1.0)))
    elevation = np.rad2deg(np.arcsin(np.clip(up / slant_range, -1.0, 1.0)))
    return slant_range, beam_separation, elevation


def load_historical_tles():
    with gzip.open(HISTORY_PATH, 'rt') as handle:
        frame = pd.DataFrame(json.load(handle))
    frame['EPOCH_DT'] = pd.to_datetime(frame.EPOCH, utc=True)
    frame['NORAD_CAT_ID'] = pd.to_numeric(
        frame.NORAD_CAT_ID, errors='coerce'
    ).astype('Int64')
    for column in ('PERIAPSIS', 'APOAPSIS'):
        frame[column] = pd.to_numeric(frame[column], errors='coerce')
    return frame.dropna(
        subset=['NORAD_CAT_ID', 'TLE_LINE1', 'TLE_LINE2', 'EPOCH_DT']
    )


def latest_prior_tle_at(history, block_start):
    prior = (
        history.loc[history.EPOCH_DT <= block_start]
        .sort_values('EPOCH_DT')
        .groupby('NORAD_CAT_ID', as_index=False)
        .tail(1)
    )
                                                                              
                                    
    return prior.loc[
        (prior.PERIAPSIS <= 1500.0) & (prior.APOAPSIS >= 600.0)
    ].reset_index(drop=True)


def refine_catalog_passage(satellite, center_time, metadata):
    start = center_time - pd.Timedelta(seconds=30)
    periods = int(60.0 / CATALOG_REFINE_STEP_SECONDS) + 1
    times = pd.date_range(
        start,
        periods=periods,
        freq=pd.to_timedelta(CATALOG_REFINE_STEP_SECONDS, unit='s'),
    )
    whole, fraction = datetime_jd(times)
    errors, positions, _ = satellite.sgp4_array(whole, fraction)
    good = errors == 0
    if not good.any():
        return None
    slant_range, beam_separation, elevation = look_from_teme(
        positions, whole + fraction
    )
    in_processed_window = (
        good
        & (slant_range >= PROCESSED_RANGE_MIN_KM)
        & (slant_range <= PROCESSED_RANGE_MAX_KM)
        & (elevation > 0.0)
    )
    if not in_processed_window.any():
        return None
    possible = np.where(in_processed_window)[0]
    selected = possible[np.argmin(beam_separation[possible])]
    if beam_separation[selected] > PRIMARY_BEAM_GATE_DEG:
        return None
    return {
        **metadata,
        'passage_time_utc': times[selected].isoformat(),
        'predicted_slant_range_km': float(slant_range[selected]),
        'predicted_beam_separation_deg': float(beam_separation[selected]),
        'predicted_elevation_deg': float(elevation[selected]),
    }


def find_catalog_passages_in_block(prior, block_start, block_end):
    times = pd.date_range(
        block_start,
        block_end,
        freq=pd.to_timedelta(CATALOG_COARSE_STEP_SECONDS, unit='s'),
        inclusive='left',
    )
    whole, fraction = datetime_jd(times)
    full_julian_date = whole + fraction
    rows = []
    chunk_size = 400
    for chunk_start in range(0, len(prior), chunk_size):
        chunk = prior.iloc[chunk_start:chunk_start + chunk_size]
        satellites, metadata_rows = [], []
        for _, row in chunk.iterrows():
            try:
                satellites.append(Satrec.twoline2rv(row.TLE_LINE1, row.TLE_LINE2))
                metadata_rows.append(row)
            except Exception:
                continue
        if not satellites:
            continue
        array = SatrecArray(satellites)
        errors, positions, _ = array.sgp4(whole, fraction)
        slant_range, beam_separation, elevation = look_from_teme(
            positions, full_julian_date[None, :]
        )
                                                                               
        broad = (
            (errors == 0)
            & (slant_range >= 800.0)
            & (slant_range <= 1300.0)
            & (beam_separation <= 20.0)
            & (elevation > 40.0)
        )
        for local_index, row in enumerate(metadata_rows):
            candidate_indices = np.where(broad[local_index])[0]
            if not len(candidate_indices):
                continue
            groups = np.split(
                candidate_indices,
                np.where(np.diff(candidate_indices) > 1)[0] + 1,
            )
            for group in groups:
                coarse_index = group[
                    np.argmin(beam_separation[local_index, group])
                ]
                metadata = {
                    'NORAD_CAT_ID': int(row.NORAD_CAT_ID),
                    'OBJECT_NAME': row.OBJECT_NAME,
                    'OBJECT_TYPE': row.OBJECT_TYPE,
                    'RCS_SIZE': row.RCS_SIZE,
                    'TLE_EPOCH_UTC': row.EPOCH_DT.isoformat(),
                    'TLE_age_at_block_start_hours': float(
                        (block_start - row.EPOCH_DT).total_seconds() / 3600.0
                    ),
                    'block_index': int(
                        (block_start - CAMPAIGN_START).total_seconds()
                        / (3600.0 * CATALOG_BLOCK_HOURS)
                    ),
                    'block_start_utc': block_start.isoformat(),
                }
                result = refine_catalog_passage(
                    satellites[local_index], times[coarse_index], metadata
                )
                if result is not None:
                    rows.append(result)
    return rows


def collapse_catalog_passages(frame, tolerance_seconds=90.0):
    if frame.empty:
        return frame.copy()
    frame = frame.copy()
    frame['passage_time'] = pd.to_datetime(
        frame.passage_time_utc, utc=True, format='mixed'
    )
    kept = []
    for _, group in frame.sort_values('passage_time').groupby('NORAD_CAT_ID'):
        cluster = []
        for index, row in group.iterrows():
            if (
                not cluster
                or (row.passage_time - frame.loc[cluster[-1], 'passage_time'])
                .total_seconds() > tolerance_seconds
            ):
                cluster.append(index)
            elif (
                row.predicted_beam_separation_deg
                < frame.loc[cluster[-1], 'predicted_beam_separation_deg']
            ):
                cluster[-1] = index
        kept.extend(cluster)
    return frame.loc[kept].sort_values('passage_time').reset_index(drop=True)


def collapse_associated_events(tier, tolerance_seconds=90.0):
    subset = events.loc[events.association_tier == tier].copy()
    subset['event_time'] = pd.to_datetime(
        subset.reference_time_utc, utc=True, format='mixed'
    )
    subset['NORAD_CAT_ID'] = pd.to_numeric(
        subset.final_NORAD_CAT_ID, errors='coerce'
    ).astype('Int64')
    subset = subset.dropna(subset=['NORAD_CAT_ID']).sort_values('event_time')
    kept = []
    for _, group in subset.groupby('NORAD_CAT_ID'):
        last_time = None
        for index, row in group.iterrows():
            if (
                last_time is None
                or (row.event_time - last_time).total_seconds() > tolerance_seconds
            ):
                kept.append(index)
                last_time = row.event_time
    return subset.loc[kept].reset_index(drop=True)


def one_to_one_match_passages(passages, detections, tolerance_seconds=120.0):
    matched_passages = np.zeros(len(passages), dtype=bool)
    matched_detection_indices = set()
    matched_event_ids = np.repeat('', len(passages)).astype(object)
    time_residual_seconds = np.repeat(np.nan, len(passages))
    for norad_id, passage_group in passages.groupby('NORAD_CAT_ID'):
        detection_group = detections.loc[detections.NORAD_CAT_ID == norad_id]
        candidates = []
        for passage_index, passage in passage_group.iterrows():
            for detection_index, detection in detection_group.iterrows():
                difference = abs(
                    (detection.event_time - passage.passage_time).total_seconds()
                )
                if difference <= tolerance_seconds:
                    candidates.append((difference, passage_index, detection_index))
        for difference, passage_index, detection_index in sorted(candidates):
            if matched_passages[passage_index] or detection_index in matched_detection_indices:
                continue
            matched_passages[passage_index] = True
            matched_detection_indices.add(detection_index)
            matched_event_ids[passage_index] = str(
                detections.loc[detection_index, 'event_id']
            )
            time_residual_seconds[passage_index] = float(difference)
    return (
        matched_passages,
        matched_event_ids,
        time_residual_seconds,
        matched_detection_indices,
    )


def wilson_interval(successes, total, confidence=0.95):
    if total <= 0:
        return np.nan, np.nan
    z = norm.ppf(0.5 + confidence / 2.0)
    proportion = successes / total
    denominator = 1.0 + z**2 / total
    center = (proportion + z**2 / (2.0 * total)) / denominator
    half = z * np.sqrt(
        proportion * (1.0 - proportion) / total + z**2 / (4.0 * total**2)
    ) / denominator
    return float(center - half), float(center + half)


catalog_cache_path = OUTPUT_DIR / 'predicted_catalog_passages.csv.gz'
catalog_audit_started = time.time()
if RUN_CATALOG_AUDIT and (FORCE_RECOMPUTE or not catalog_cache_path.exists()):
    historical_tles = load_historical_tles()
    passage_rows = []
    for block_index in range(
        int((CAMPAIGN_END - CAMPAIGN_START).total_seconds() / 3600)
        // CATALOG_BLOCK_HOURS
    ):
        block_start = CAMPAIGN_START + pd.Timedelta(
            hours=CATALOG_BLOCK_HOURS * block_index
        )
        block_end = block_start + pd.Timedelta(hours=CATALOG_BLOCK_HOURS)
        prior = latest_prior_tle_at(historical_tles, block_start)
        print(
            f'Catalog block {block_index + 1}/4: propagating {len(prior):,} objects',
            flush=True,
        )
        block_rows = find_catalog_passages_in_block(prior, block_start, block_end)
        print('  refined passages before boundary collapse:', len(block_rows), flush=True)
        passage_rows.extend(block_rows)
    catalog_passages = collapse_catalog_passages(pd.DataFrame(passage_rows))
else:
    if not catalog_cache_path.exists():
        raise RuntimeError(
            'Catalog audit was disabled but no cached passage audit exists.'
        )
    catalog_passages = pd.read_csv(catalog_cache_path)
    catalog_passages['passage_time'] = pd.to_datetime(
        catalog_passages.passage_time_utc, utc=True, format='mixed'
    )

strict_passes = collapse_associated_events('tier_a_strict')
supported_passes = collapse_associated_events('tier_b_empirical')
(
    strict_detected,
    strict_event_ids,
    strict_time_residual,
    matched_strict_detection_indices,
) = one_to_one_match_passages(catalog_passages, strict_passes)
(
    supported_detected,
    supported_event_ids,
    supported_time_residual,
    matched_supported_detection_indices,
) = one_to_one_match_passages(catalog_passages, supported_passes)

catalog_passages['strict_detected'] = strict_detected
catalog_passages['strict_event_id'] = strict_event_ids
catalog_passages['strict_time_residual_seconds'] = strict_time_residual
catalog_passages['supported_detected'] = supported_detected
catalog_passages['supported_event_id'] = supported_event_ids
catalog_passages['supported_time_residual_seconds'] = supported_time_residual
catalog_passages['RCS_SIZE'] = (
    catalog_passages.RCS_SIZE.fillna('UNKNOWN').astype(str).str.upper()
)
catalog_passages.drop(columns=['passage_time'], errors='ignore').to_csv(
    catalog_cache_path, index=False, compression='gzip'
)

catalog_recovery_rows = []
for label, group in [('ALL', catalog_passages), *catalog_passages.groupby('RCS_SIZE')]:
    successes = int(group.strict_detected.sum())
    total = int(len(group))
    low, high = wilson_interval(successes, total)
    catalog_recovery_rows.append({
        'RCS_SIZE': label,
        'predicted_passages': total,
        'strict_detected_passages': successes,
        'strict_recovery_fraction': successes / total if total else np.nan,
        'wilson_ci95_low': low,
        'wilson_ci95_high': high,
    })
catalog_recovery_by_rcs = pd.DataFrame(catalog_recovery_rows)

beam_gate_rows = []
for gate in BEAM_GATE_SENSITIVITY_DEG:
    gate_frame = catalog_passages.loc[
        catalog_passages.predicted_beam_separation_deg <= gate
    ]
    for label, group in [('ALL', gate_frame), *gate_frame.groupby('RCS_SIZE')]:
        successes = int(group.strict_detected.sum())
        total = int(len(group))
        low, high = wilson_interval(successes, total)
        beam_gate_rows.append({
            'beam_gate_deg': gate,
            'RCS_SIZE': label,
            'predicted_passages': total,
            'strict_detected_passages': successes,
            'strict_recovery_fraction': successes / total if total else np.nan,
            'wilson_ci95_low': low,
            'wilson_ci95_high': high,
        })
beam_gate_sensitivity = pd.DataFrame(beam_gate_rows)

catalog_block_recovery = (
    catalog_passages.groupby('block_index')
    .strict_detected.agg(['sum', 'count', 'mean'])
    .rename(columns={
        'sum': 'strict_detected_passages',
        'count': 'predicted_passages',
        'mean': 'strict_recovery_fraction',
    })
    .reset_index()
)

                                                                              
                                                                                 
def passage_design_matrix(frame):
    rcs = frame.RCS_SIZE.fillna('UNKNOWN').astype(str).str.upper()
    return np.column_stack([
        np.ones(len(frame)),
        (frame.predicted_beam_separation_deg.to_numpy() - 2.5) / 2.5,
        (frame.predicted_slant_range_km.to_numpy() - 1055.0) / 30.0,
        np.clip(frame.TLE_age_at_block_start_hours.to_numpy(), 0.0, 120.0) / 24.0,
        (rcs == 'MEDIUM').astype(float),
        (rcs == 'LARGE').astype(float),
    ])


def fit_penalized_logistic(design, outcomes, penalty=1.0):
    outcomes = np.asarray(outcomes, dtype=float)
    penalty_weights = np.r_[0.0, np.repeat(1.0, design.shape[1] - 1)]

    def objective_gradient(coefficients):
        eta = design @ coefficients
        objective = np.logaddexp(0.0, eta).sum() - outcomes @ eta
        objective += 0.5 * penalty * np.sum(
            penalty_weights * coefficients**2
        )
        probabilities = 1.0 / (1.0 + np.exp(-np.clip(eta, -30.0, 30.0)))
        gradient = design.T @ (probabilities - outcomes)
        gradient += penalty * penalty_weights * coefficients
        return float(objective), gradient

    prevalence = np.clip(outcomes.mean(), 1e-4, 1.0 - 1e-4)
    initial = np.zeros(design.shape[1])
    initial[0] = np.log(prevalence / (1.0 - prevalence))
    result = minimize(
        objective_gradient, initial, jac=True, method='L-BFGS-B',
        options={'maxiter': 200, 'ftol': 1e-12, 'gtol': 1e-8},
    )
    return result.x, bool(result.success)


catalog_predictions = []
for held_out_block in sorted(catalog_passages.block_index.unique()):
    training = catalog_passages.loc[
        catalog_passages.block_index != held_out_block
    ]
    testing = catalog_passages.loc[
        catalog_passages.block_index == held_out_block
    ]
    training_y = training.strict_detected.astype(int).to_numpy()
    coefficients, converged = fit_penalized_logistic(
        passage_design_matrix(training), training_y
    )
    test_eta = passage_design_matrix(testing) @ coefficients
    probabilities = 1.0 / (1.0 + np.exp(-np.clip(test_eta, -30.0, 30.0)))
    baseline_probability = float(training_y.mean())
    for (_, passage), probability in zip(testing.iterrows(), probabilities):
        catalog_predictions.append({
            'held_out_block': int(held_out_block),
            'NORAD_CAT_ID': int(passage.NORAD_CAT_ID),
            'passage_time_utc': passage.passage_time_utc,
            'RCS_SIZE': passage.RCS_SIZE,
            'strict_detected': int(passage.strict_detected),
            'predicted_detection_probability': float(probability),
            'training_prevalence_baseline': baseline_probability,
            'optimizer_success': converged,
        })
catalog_heldout_predictions = pd.DataFrame(catalog_predictions)
catalog_y = catalog_heldout_predictions.strict_detected.to_numpy(dtype=float)
catalog_probability = np.clip(
    catalog_heldout_predictions.predicted_detection_probability.to_numpy(),
    1e-8, 1.0 - 1e-8,
)
catalog_baseline_probability = np.clip(
    catalog_heldout_predictions.training_prevalence_baseline.to_numpy(),
    1e-8, 1.0 - 1e-8,
)
catalog_heldout_validation = {
    'folds': int(catalog_heldout_predictions.held_out_block.nunique()),
    'passages': int(len(catalog_heldout_predictions)),
    'brier_score': float(np.mean((catalog_y - catalog_probability)**2)),
    'prevalence_baseline_brier_score': float(
        np.mean((catalog_y - catalog_baseline_probability)**2)
    ),
    'log_loss': float(-np.mean(
        catalog_y * np.log(catalog_probability)
        + (1.0 - catalog_y) * np.log(1.0 - catalog_probability)
    )),
    'prevalence_baseline_log_loss': float(-np.mean(
        catalog_y * np.log(catalog_baseline_probability)
        + (1.0 - catalog_y) * np.log(1.0 - catalog_baseline_probability)
    )),
    'optimizer_success_all_folds': bool(
        catalog_heldout_predictions.optimizer_success.all()
    ),
}

catalog_validation_summary = {
    'predicted_catalog_passages': int(len(catalog_passages)),
    'strict_associated_events_before_pass_collapse': int(
        (events.association_tier == 'tier_a_strict').sum()
    ),
    'unique_strict_detected_passes': int(len(strict_passes)),
    'strict_detected_passages_recovered_by_catalog_first_audit': int(
        catalog_passages.strict_detected.sum()
    ),
    'strict_detected_passes_not_found_by_audit': int(
        len(strict_passes) - len(matched_strict_detection_indices)
    ),
    'unique_supported_detected_passes': int(len(supported_passes)),
    'supported_passages_recovered_by_catalog_first_audit': int(
        catalog_passages.supported_detected.sum()
    ),
    'overall_strict_recovery_fraction': float(
        catalog_passages.strict_detected.mean()
    ),
    'minimum_tle_age_at_block_start_hours': float(
        catalog_passages.TLE_age_at_block_start_hours.min()
    ),
    'maximum_tle_age_at_block_start_hours': float(
        catalog_passages.TLE_age_at_block_start_hours.max()
    ),
    'primary_beam_gate_deg': PRIMARY_BEAM_GATE_DEG,
    'processed_range_min_km': PROCESSED_RANGE_MIN_KM,
    'processed_range_max_km': PROCESSED_RANGE_MAX_KM,
    'heldout_prediction': catalog_heldout_validation,
    'runtime_seconds': float(time.time() - catalog_audit_started),
}

print('\nCATALOG-FIRST VALIDATION COMPLETE')
display(pd.DataFrame([catalog_validation_summary]).drop(
    columns=['heldout_prediction']
))
display(catalog_recovery_by_rcs)
print('Held-out six-hour-block prediction:')
display(pd.DataFrame([catalog_heldout_validation]))


## 7. Claim gates


In [ ]:
FULL_PUBLICATION_RUN = bool(
    N_MAIN_REPLICATES >= 200
    and N_SENSITIVITY_REPLICATES >= 100
    and BUDGET_STEPS == 24
)


def select_final_summary(scenario, policy, experiment_block='main', strength=1.0,
                         response_correlation_deg=PRIMARY_RESPONSE_CORRELATION_DEG):
    selected = final_summary.loc[
        (final_summary.experiment_block == experiment_block)
        & (final_summary.scenario == scenario)
        & np.isclose(final_summary.strength, strength)
        & np.isclose(
            final_summary.response_correlation_deg, response_correlation_deg
        )
        & (final_summary.policy == policy)
    ]
    if len(selected) != 1:
        raise RuntimeError(
            f'Expected one summary row for {scenario}/{policy}; found {len(selected)}.'
        )
    return selected.iloc[0]


def select_pairwise(scenario, baseline, experiment_block='main', strength=1.0,
                    response_correlation_deg=PRIMARY_RESPONSE_CORRELATION_DEG):
    selected = pairwise.loc[
        (pairwise.experiment_block == experiment_block)
        & (pairwise.scenario == scenario)
        & np.isclose(pairwise.strength, strength)
        & np.isclose(pairwise.response_correlation_deg, response_correlation_deg)
        & (pairwise.baseline == baseline)
    ]
    if len(selected) != 1:
        raise RuntimeError(
            f'Expected one pairwise row for {scenario}/{baseline}; found {len(selected)}.'
        )
    return selected.iloc[0]


reported_asset = select_final_summary(
    'reported_response_uncertainty', 'asset_aopt'
)
hard_asset = select_final_summary(
    'full_cell_response_perturbation', 'asset_aopt'
)
reported_fixed = select_pairwise(
    'reported_response_uncertainty', 'best_fixed'
)
reported_random = select_pairwise(
    'reported_response_uncertainty', 'random'
)
hard_random = select_pairwise(
    'full_cell_response_perturbation', 'random'
)
reported_unweighted = select_pairwise(
    'reported_response_uncertainty', 'unweighted_aopt'
)

claim_gate_rows = [
    {
        'gate': 'full_predeclared_replicate_counts',
        'primary': True,
        'passed': FULL_PUBLICATION_RUN,
        'observed': (
            f'main={N_MAIN_REPLICATES}, sensitivity={N_SENSITIVITY_REPLICATES}, '
            f'budget={BUDGET_STEPS}'
        ),
        'criterion': 'main >= 200, sensitivity >= 100, budget = 24',
    },
    {
        'gate': 'exact_optimizer_reliability',
        'primary': True,
        'passed': bool(final_summary.optimizer_failure_rate.max() <= 0.01),
        'observed': float(final_summary.optimizer_failure_rate.max()),
        'criterion': 'maximum failure rate <= 0.01',
    },
    {
        'gate': 'sequential_filter_matches_exact_final_refit',
        'primary': True,
        'passed': bool(
            final_summary.filter_exact_relative_difference_median.max() <= 0.05
        ),
        'observed': float(
            final_summary.filter_exact_relative_difference_median.max()
        ),
        'criterion': 'maximum condition-level median relative difference <= 0.05',
    },
    {
        'gate': 'reported_response_95_coverage',
        'primary': True,
        'passed': bool(
            0.90 <= reported_asset.cellwise_95_coverage_mean <= 0.98
        ),
        'observed': float(reported_asset.cellwise_95_coverage_mean),
        'criterion': '0.90 <= mean empirical coverage <= 0.98',
    },
    {
        'gate': 'hard_response_95_coverage',
        'primary': True,
        'passed': bool(0.90 <= hard_asset.cellwise_95_coverage_mean <= 0.98),
        'observed': float(hard_asset.cellwise_95_coverage_mean),
        'criterion': '0.90 <= mean empirical coverage <= 0.98',
    },
    {
        'gate': 'adaptive_beats_best_fixed_uncertainty',
        'primary': True,
        'passed': bool(reported_fixed.uncertainty_improvement_ci95_low > 0.0),
        'observed': float(reported_fixed.uncertainty_improvement_ci95_low),
        'criterion': 'paired 95% CI lower bound > 0%',
    },
    {
        'gate': 'adaptive_beats_best_fixed_rmse',
        'primary': True,
        'passed': bool(reported_fixed.rmse_improvement_ci95_low > 0.0),
        'observed': float(reported_fixed.rmse_improvement_ci95_low),
        'criterion': 'paired 95% CI lower bound > 0%',
    },
    {
        'gate': 'adaptive_beats_random_uncertainty',
        'primary': True,
        'passed': bool(reported_random.uncertainty_improvement_ci95_low > 0.0),
        'observed': float(reported_random.uncertainty_improvement_ci95_low),
        'criterion': 'paired 95% CI lower bound > 0%',
    },
    {
        'gate': 'hard_response_rmse_noninferior_to_random_within_5_percent',
        'primary': True,
        'passed': bool(
            hard_random.rmse_raw_difference_ci95_low
            > -RMSE_NONINFERIORITY_MARGIN_RELATIVE
            * select_final_summary(
                'full_cell_response_perturbation', 'random'
            ).asset_weighted_rmse_mean
        ),
        'observed': float(hard_random.rmse_raw_difference_ci95_low),
        'criterion': (
            'paired raw-difference 95% CI lower bound exceeds -5% of random mean RMSE'
        ),
    },
    {
        'gate': 'catalog_audit_finds_all_unique_strict_passes',
        'primary': True,
        'passed': bool(
            catalog_validation_summary[
                'strict_detected_passes_not_found_by_audit'
            ] == 0
        ),
        'observed': int(
            catalog_validation_summary['strict_detected_passes_not_found_by_audit']
        ),
        'criterion': 'zero unique strict detected passes absent from audit',
    },
    {
        'gate': 'catalog_audit_has_large_denominator',
        'primary': True,
        'passed': bool(len(catalog_passages) >= 150),
        'observed': int(len(catalog_passages)),
        'criterion': 'at least 150 predicted passages',
    },
    {
        'gate': 'catalog_uses_no_future_tles',
        'primary': True,
        'passed': bool(
            catalog_passages.TLE_age_at_block_start_hours.min() >= -1e-9
        ),
        'observed': float(
            catalog_passages.TLE_age_at_block_start_hours.min()
        ),
        'criterion': 'minimum TLE age at block start >= 0 hours',
    },
    {
        'gate': 'asset_weighting_beats_unweighted_adaptive',
        'primary': False,
        'passed': bool(
            reported_unweighted.uncertainty_improvement_ci95_low > 0.0
        ),
        'observed': float(
            reported_unweighted.uncertainty_improvement_ci95_low
        ),
        'criterion': 'paired uncertainty 95% CI lower bound > 0%',
    },
    {
        'gate': 'hard_response_rmse_superiority_over_random',
        'primary': False,
        'passed': bool(hard_random.rmse_raw_difference_ci95_low > 0.0),
        'observed': float(hard_random.rmse_raw_difference_ci95_low),
        'criterion': 'paired raw-difference 95% CI lower bound > 0',
    },
    {
        'gate': 'catalog_heldout_model_beats_prevalence_brier',
        'primary': False,
        'passed': bool(
            catalog_heldout_validation['brier_score']
            < catalog_heldout_validation['prevalence_baseline_brier_score']
        ),
        'observed': float(catalog_heldout_validation['brier_score']),
        'criterion': 'held-out Brier score lower than prevalence baseline',
    },
]
claim_gates = pd.DataFrame(claim_gate_rows)
PRIMARY_GATES_PASSED = bool(
    claim_gates.loc[claim_gates.primary, 'passed'].all()
)
ASSET_WEIGHTING_SUPERIORITY_SUPPORTED = bool(
    claim_gates.loc[
        claim_gates.gate == 'asset_weighting_beats_unweighted_adaptive',
        'passed',
    ].iloc[0]
)

claim_status = {
    'full_publication_run': FULL_PUBLICATION_RUN,
    'all_primary_claim_gates_passed': PRIMARY_GATES_PASSED,
    'asset_weighting_superiority_supported': ASSET_WEIGHTING_SUPERIORITY_SUPPORTED,
    'central_claim_if_primary_gates_pass': (
        'Under equal simulated radar time, response-uncertainty-aware adaptive '
        'pointing reduces satellite-exposure-weighted population uncertainty '
        'relative to the strongest fixed elevation and random schedules, improves '
        'accuracy over the strongest fixed elevation, and maintains near-nominal '
        'interval coverage when ORDEM radar responses are imperfect. Under the '
        'hardest structural response mismatch, its RMSE is noninferior to random '
        'within the explicit 5% practical margin; superiority to random is not claimed.'
    ),
    'asset_weighting_claim_rule': (
        'Claim that satellite weighting itself improves overall reconstruction only '
        'when the separate ablation gate passes.'
    ),
    'prohibited_claims': [
        'historical EISCAT radar was adaptively steered',
        'individual debris orbits were reconstructed',
        'collision warnings were produced',
        'unresolved detections are proven uncataloged debris',
        'equal active-satellite exposure is a validated critical-infrastructure score',
    ],
    'external_validation_status': (
        'Svalbard raw radar data were not present in the frozen project inputs. '
        'The catalog-first Tromso audit is a real-data internal validation, not a '
        'claim of independent twin-radar replication.'
    ),
}

print('\nCLAIM-GATE RESULTS')
display(claim_gates)
print('Primary publication gates passed:', PRIMARY_GATES_PASSED)
print('Asset-weighting superiority supported:', ASSET_WEIGHTING_SUPERIORITY_SUPPORTED)


## 8. Manuscript figures


In [ ]:
plt.rcParams.update({
    'figure.dpi': 125,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'legend.fontsize': 8,
    'axes.spines.top': False,
    'axes.spines.right': False,
})


def reported_curve(metric):
    return curve_summary.loc[
        (curve_summary.experiment_block == 'main')
        & (curve_summary.scenario == 'reported_response_uncertainty')
        & np.isclose(curve_summary.strength, 1.0)
        & np.isclose(
            curve_summary.response_correlation_deg,
            PRIMARY_RESPONSE_CORRELATION_DEG,
        )
    ].copy()


def plot_policy_curve(axis, metric, ylabel, title):
    frame = reported_curve(metric)
    for policy in POLICIES:
        group = frame.loc[frame.policy == policy].sort_values('step')
        axis.plot(
            group.step,
            group[f'{metric}_mean'],
            label=POLICY_LABELS[policy],
            color=POLICY_COLORS[policy],
            linewidth=2.2 if policy == 'asset_aopt' else 1.35,
            alpha=1.0 if policy in {'asset_aopt', 'best_fixed', 'random'} else 0.82,
        )
        axis.fill_between(
            group.step,
            group[f'{metric}_ci95_low'],
            group[f'{metric}_ci95_high'],
            color=POLICY_COLORS[policy],
            alpha=0.08,
            linewidth=0,
        )
    axis.set(xlabel='Radar hours used', ylabel=ylabel, title=title)
    axis.set_xlim(0, BUDGET_STEPS)
    axis.grid(alpha=0.2)


figure_main, axes = plt.subplots(2, 2, figsize=(12.0, 8.5), constrained_layout=True)
plot_policy_curve(
    axes[0, 0], 'asset_uncertainty_fraction',
    'Uncertainty remaining',
    'A. Learning under reported ORDEM response uncertainty',
)
plot_policy_curve(
    axes[0, 1], 'asset_weighted_rmse',
    'Reconstruction RMSE',
    'B. Accuracy as observations accumulate',
)
axes[0, 0].legend(ncol=2, frameon=False, loc='best')

paired_reported = pairwise.loc[
    (pairwise.experiment_block == 'main')
    & (pairwise.scenario == 'reported_response_uncertainty')
    & np.isclose(pairwise.strength, 1.0)
    & np.isclose(
        pairwise.response_correlation_deg, PRIMARY_RESPONSE_CORRELATION_DEG
    )
].set_index('baseline').reindex([
    'best_fixed', 'random', 'unweighted_aopt', 'eig'
]).reset_index()
x_locations = np.arange(len(paired_reported))
means = paired_reported.uncertainty_improvement_percent_mean.to_numpy()
low = paired_reported.uncertainty_improvement_ci95_low.to_numpy()
high = paired_reported.uncertainty_improvement_ci95_high.to_numpy()
axes[1, 0].errorbar(
    x_locations, means,
    yerr=np.vstack([
        np.maximum(means - low, 0.0),
        np.maximum(high - means, 0.0),
    ]),
    fmt='o', color=POLICY_COLORS['asset_aopt'], capsize=4, linewidth=1.6,
)
axes[1, 0].axhline(0.0, color='black', linewidth=0.9, linestyle='--')
axes[1, 0].set_xticks(x_locations)
axes[1, 0].set_xticklabels([
    POLICY_LABELS[baseline] for baseline in paired_reported.baseline
], rotation=18, ha='right')
axes[1, 0].set(
    ylabel='Uncertainty reduction vs baseline (%)',
    title='C. Paired final advantage of satellite-weighted adaptive pointing',
)
axes[1, 0].grid(axis='y', alpha=0.2)

reported_actions = action_history.loc[
    (action_history.experiment_block == 'main')
    & (action_history.scenario == 'reported_response_uncertainty')
    & np.isclose(action_history.strength, 1.0)
    & np.isclose(
        action_history.response_correlation_deg, PRIMARY_RESPONSE_CORRELATION_DEG
    )
]
for policy in POLICIES:
    group = reported_actions.loc[reported_actions.policy == policy]
    frequencies = (
        group.elevation_deg.value_counts(normalize=True)
        .reindex(ELEVATIONS, fill_value=0.0)
    )
    axes[1, 1].plot(
        ELEVATIONS, frequencies,
        color=POLICY_COLORS[policy],
        label=POLICY_LABELS[policy],
        linewidth=2.2 if policy == 'asset_aopt' else 1.25,
        alpha=1.0 if policy in {'asset_aopt', 'best_fixed', 'random'} else 0.8,
    )
axes[1, 1].set(
    xlabel='Selected elevation (degrees)',
    ylabel='Fraction of observations',
    title='D. The policies use the fine elevation grid differently',
)
axes[1, 1].set_xlim(ELEVATIONS.min(), ELEVATIONS.max())
axes[1, 1].grid(alpha=0.2)

main_png = OUTPUT_DIR / 'figure_policy_response_robustness.png'
main_pdf = OUTPUT_DIR / 'figure_policy_response_robustness.pdf'
figure_main.savefig(main_png, bbox_inches='tight')
figure_main.savefig(main_pdf, bbox_inches='tight')
plt.show()

                                                                                   
try:
    catalog_heldout_predictions['probability_bin'] = pd.qcut(
        catalog_heldout_predictions.predicted_detection_probability,
        q=5,
        duplicates='drop',
    )
except ValueError:
    catalog_heldout_predictions['probability_bin'] = pd.cut(
        catalog_heldout_predictions.predicted_detection_probability,
        bins=np.linspace(0.0, 1.0, 6),
        include_lowest=True,
    )
catalog_calibration_rows = []
for _, group in catalog_heldout_predictions.groupby(
    'probability_bin', observed=True
):
    observed = int(group.strict_detected.sum())
    total = int(len(group))
    low, high = wilson_interval(observed, total)
    catalog_calibration_rows.append({
        'mean_predicted_probability': float(
            group.predicted_detection_probability.mean()
        ),
        'observed_recovery_fraction': observed / total,
        'ci95_low': low,
        'ci95_high': high,
        'passages': total,
    })
catalog_calibration = pd.DataFrame(catalog_calibration_rows)

figure_validation, axes = plt.subplots(
    2, 3, figsize=(15.0, 8.7), constrained_layout=True
)
rcs_order = [
    value for value in ('SMALL', 'MEDIUM', 'LARGE', 'UNKNOWN')
    if value in set(catalog_passages.RCS_SIZE)
]
rcs_counts = catalog_recovery_by_rcs.loc[
    catalog_recovery_by_rcs.RCS_SIZE.isin(rcs_order)
].set_index('RCS_SIZE').reindex(rcs_order)
x_locations = np.arange(len(rcs_order))
axes[0, 0].bar(
    x_locations - 0.18, rcs_counts.predicted_passages,
    width=0.36, label='Predicted passages', color='#9ecae9',
)
axes[0, 0].bar(
    x_locations + 0.18, rcs_counts.strict_detected_passages,
    width=0.36, label='Strictly recovered', color='#2171b5',
)
axes[0, 0].set_xticks(x_locations)
axes[0, 0].set_xticklabels([value.title() for value in rcs_order])
axes[0, 0].set(
    ylabel='Unique passages',
    title='A. Catalog-first detector recovery denominator',
)
axes[0, 0].legend(frameon=False)

recovery_means = rcs_counts.strict_recovery_fraction.to_numpy()
recovery_low = rcs_counts.wilson_ci95_low.to_numpy()
recovery_high = rcs_counts.wilson_ci95_high.to_numpy()
axes[0, 1].errorbar(
    x_locations, recovery_means,
    yerr=np.vstack([
        np.maximum(recovery_means - recovery_low, 0.0),
        np.maximum(recovery_high - recovery_means, 0.0),
    ]),
    fmt='o', color='#2171b5', capsize=4, markersize=7,
)
axes[0, 1].set_xticks(x_locations)
axes[0, 1].set_xticklabels([value.title() for value in rcs_order])
axes[0, 1].set_ylim(0.0, min(1.0, max(0.6, recovery_high.max() + 0.08)))
axes[0, 1].set(
    ylabel='Strict recovery fraction',
    title='B. Recovery depends strongly on catalog RCS class',
)
axes[0, 1].grid(axis='y', alpha=0.2)

axes[0, 2].errorbar(
    catalog_calibration.mean_predicted_probability,
    catalog_calibration.observed_recovery_fraction,
    yerr=np.vstack([
        np.maximum(
            catalog_calibration.observed_recovery_fraction
            - catalog_calibration.ci95_low,
            0.0,
        ),
        np.maximum(
            catalog_calibration.ci95_high
            - catalog_calibration.observed_recovery_fraction,
            0.0,
        ),
    ]),
    fmt='o', color='#756bb1', capsize=3,
)
axes[0, 2].plot([0, 1], [0, 1], linestyle='--', color='black', linewidth=1)
axes[0, 2].set(
    xlim=(0, 0.75), ylim=(0, 0.75),
    xlabel='Predicted recovery probability',
    ylabel='Observed recovery fraction',
    title='C. Prediction on untouched six-hour blocks',
)
axes[0, 2].grid(alpha=0.2)

coverage_plot = coverage_reliability.loc[
    (coverage_reliability.experiment_block == 'main')
    & (coverage_reliability.scenario.isin([
        'reported_response_uncertainty',
        'full_cell_response_perturbation',
    ]))
    & (coverage_reliability.policy == 'asset_aopt')
    & np.isclose(coverage_reliability.strength, 1.0)
    & np.isclose(
        coverage_reliability.response_correlation_deg,
        PRIMARY_RESPONSE_CORRELATION_DEG,
    )
]
coverage_labels = {
    'reported_response_uncertainty': 'Reported ORDEM error',
    'full_cell_response_perturbation': 'Hard response error',
}
for scenario, group in coverage_plot.groupby('scenario'):
    group = group.sort_values('nominal_level')
    axes[1, 0].plot(
        group.nominal_level, group.observed_coverage,
        marker='o', linewidth=1.8, label=coverage_labels[scenario],
    )
axes[1, 0].plot([0.45, 1.0], [0.45, 1.0], '--', color='black', linewidth=1)
axes[1, 0].set(
    xlim=(0.45, 1.0), ylim=(0.45, 1.0),
    xlabel='Claimed interval probability',
    ylabel='Observed coverage',
    title='D. Bayesian interval calibration',
)
axes[1, 0].legend(frameon=False)
axes[1, 0].grid(alpha=0.2)

sbc_plot = aggregate_sbc.loc[
    (aggregate_sbc.experiment_block == 'main')
    & (aggregate_sbc.scenario == 'full_cell_response_perturbation')
    & (aggregate_sbc.policy == 'asset_aopt')
    & (aggregate_sbc['aggregate'] == 'asset_weighted_log_multiplier')
    & np.isclose(aggregate_sbc.strength, 1.0)
    & np.isclose(
        aggregate_sbc.response_correlation_deg,
        PRIMARY_RESPONSE_CORRELATION_DEG,
    )
]
axes[1, 1].hist(
    sbc_plot['rank'] / (SBC_DRAWS + 1.0),
    bins=np.linspace(0, 1, 11), color='#74c476', edgecolor='white',
)
axes[1, 1].axhline(
    len(sbc_plot) / 10.0, linestyle='--', color='black', linewidth=1,
)
axes[1, 1].set(
    xlabel='Posterior rank fraction',
    ylabel='Independent simulated worlds',
    title='E. One aggregate SBC rank per world',
)

ablation_plot = shell_ablation.loc[
    shell_ablation.scenario == 'reported_response_uncertainty'
].sort_values('altitude_center_km')
axes[1, 2].bar(
    ablation_plot.altitude_center_km,
    ablation_plot.asset_vs_unweighted_variance_improvement_percent_mean,
    width=11.0, color='#e6550d', alpha=0.78,
)
axes[1, 2].axhline(0.0, color='black', linewidth=0.9)
axes[1, 2].set(
    xlabel='Altitude-shell center (km)',
    ylabel='Variance reduction vs unweighted (%)',
    title='F. Where satellite weighting changes learning',
)
asset_axis = axes[1, 2].twinx()
asset_axis.plot(
    ALTITUDE_CENTERS, ASSET_WEIGHTS,
    color='#3182bd', marker='o', linewidth=1.4,
)
asset_axis.set_ylabel('Active-satellite exposure weight', color='#3182bd')
asset_axis.tick_params(axis='y', colors='#3182bd')

validation_png = OUTPUT_DIR / 'figure_validation_catalog_and_calibration.png'
validation_pdf = OUTPUT_DIR / 'figure_validation_catalog_and_calibration.pdf'
figure_validation.savefig(validation_png, bbox_inches='tight')
figure_validation.savefig(validation_pdf, bbox_inches='tight')
plt.show()


## 9. Freeze outputs and print the decision


In [ ]:
output_frames = {
    'response_robust_step_metrics.csv.gz': step_metrics,
    'response_robust_action_history.csv.gz': action_history,
    'response_robust_shell_metrics.csv.gz': shell_metrics,
    'independent_aggregate_sbc_ranks.csv.gz': aggregate_sbc,
    'learning_curve_summary.csv': curve_summary,
    'final_policy_summary.csv': final_summary,
    'paired_policy_comparisons.csv': pairwise,
    'coverage_reliability.csv': coverage_reliability,
    'asset_weighting_shell_ablation.csv': shell_ablation,
    'independent_aggregate_sbc_summary.csv': aggregate_sbc_summary,
    'catalog_recovery_by_rcs.csv': catalog_recovery_by_rcs,
    'catalog_beam_gate_sensitivity.csv': beam_gate_sensitivity,
    'catalog_block_recovery.csv': catalog_block_recovery,
    'catalog_heldout_predictions.csv': catalog_heldout_predictions,
    'catalog_heldout_calibration.csv': catalog_calibration,
    'claim_gates.csv': claim_gates,
}
for filename, frame in output_frames.items():
    destination = OUTPUT_DIR / filename
    temporary = OUTPUT_DIR / f'{filename}.tmp'
    frame.to_csv(
        temporary,
        index=False,
        compression='gzip' if filename.endswith('.gz') else None,
    )
    os.replace(temporary, destination)
                                                                           
    verified_rows = len(pd.read_csv(destination))
    if verified_rows != len(frame):
        raise RuntimeError(
            f'Output verification failed for {filename}: '
            f'{verified_rows} rows read back, expected {len(frame)}.'
        )

manuscript_rows = []
for _, row in final_summary.loc[
    (final_summary.experiment_block == 'main')
    & np.isclose(final_summary.strength, 1.0)
    & np.isclose(
        final_summary.response_correlation_deg, PRIMARY_RESPONSE_CORRELATION_DEG
    )
].iterrows():
    manuscript_rows.append({
        'scenario': row.scenario,
        'policy': POLICY_LABELS[row.policy],
        'worlds': int(row.n),
        'uncertainty_remaining_mean': row.asset_uncertainty_fraction_mean,
        'uncertainty_remaining_ci95_low': row.asset_uncertainty_fraction_ci95_low,
        'uncertainty_remaining_ci95_high': row.asset_uncertainty_fraction_ci95_high,
        'rmse_mean': row.asset_weighted_rmse_mean,
        'rmse_ci95_low': row.asset_weighted_rmse_ci95_low,
        'rmse_ci95_high': row.asset_weighted_rmse_ci95_high,
        'coverage_95_mean': row.cellwise_95_coverage_mean,
        'coverage_95_ci95_low': row.cellwise_95_coverage_ci95_low,
        'coverage_95_ci95_high': row.cellwise_95_coverage_ci95_high,
    })
manuscript_key_results = pd.DataFrame(manuscript_rows)
manuscript_key_results.to_csv(
    OUTPUT_DIR / 'manuscript_key_results.csv', index=False
)

(OUTPUT_DIR / 'real_count_model_comparison.json').write_text(
    json.dumps(real_count_model_comparison, indent=2)
)
(OUTPUT_DIR / 'catalog_validation_summary.json').write_text(
    json.dumps(catalog_validation_summary, indent=2)
)
(OUTPUT_DIR / 'claim_status.json').write_text(
    json.dumps(claim_status, indent=2)
)

readme_lines = [
    '# research publication-lock validation outputs',
    '',
    f'- Full publication run: {FULL_PUBLICATION_RUN}',
    f'- All primary claim gates passed: {PRIMARY_GATES_PASSED}',
    f'- Asset-weighting superiority supported: {ASSET_WEIGHTING_SUPERIORITY_SUPPORTED}',
    f"- Predicted catalog passages: {catalog_validation_summary['predicted_catalog_passages']}",
    f"- Unique strict passes recovered: {catalog_validation_summary['strict_detected_passages_recovered_by_catalog_first_audit']}",
    '',
    '## Interpretation',
    '',
    claim_status['central_claim_if_primary_gates_pass'],
    '',
    claim_status['asset_weighting_claim_rule'],
    '',
    'The catalog audit validates detector recovery and misses for known catalog objects. '
    'It does not prove that unresolved events are uncataloged debris.',
    '',
    'Independent Svalbard raw radar data were not available in the frozen inputs. '
    'Do not describe this run as twin-radar replication.',
]
(OUTPUT_DIR / 'README.md').write_text('\n'.join(readme_lines) + '\n')

manifest_entries = []
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file() and path.name not in {'output_manifest.json', '_SUCCESS.json'}:
        manifest_entries.append({
            'filename': path.name,
            'bytes': path.stat().st_size,
            'sha256': sha256_file(path),
        })
manifest = {
    'configuration_hash': config_hash,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'full_publication_run': FULL_PUBLICATION_RUN,
    'all_primary_claim_gates_passed': PRIMARY_GATES_PASSED,
    'files': manifest_entries,
}
(OUTPUT_DIR / 'output_manifest.json').write_text(
    json.dumps(manifest, indent=2)
)

success_payload = {
    'status': 'complete',
    'configuration_hash': config_hash,
    'full_publication_run': FULL_PUBLICATION_RUN,
    'all_primary_claim_gates_passed': PRIMARY_GATES_PASSED,
    'asset_weighting_superiority_supported': ASSET_WEIGHTING_SUPERIORITY_SUPPORTED,
    'output_directory': str(OUTPUT_DIR),
    'finished_utc': datetime.now(timezone.utc).isoformat(),
}
(OUTPUT_DIR / '_SUCCESS.json').write_text(
    json.dumps(success_payload, indent=2)
)

print('\n' + '=' * 72)
print('SUCCESS — PUBLICATION-LOCK NOTEBOOK COMPLETED')
if not FULL_PUBLICATION_RUN:
    print('TEST MODE ONLY — run the default 200/100 replicate configuration before use.')
elif PRIMARY_GATES_PASSED:
    print('PRIMARY CLAIM GATES PASSED — results may be frozen for the manuscript.')
else:
    print('PRIMARY CLAIM GATES DID NOT ALL PASS — do not freeze the manuscript yet.')
    print('Failed primary gates:')
    display(claim_gates.loc[claim_gates.primary & ~claim_gates.passed])
if not ASSET_WEIGHTING_SUPERIORITY_SUPPORTED:
    print(
        'The separate asset-weighting superiority claim is not supported. '
        'Frame weighting as a transparent objective, not as a proven overall advantage.'
    )
print('Outputs:', OUTPUT_DIR)
print('=' * 72)
